In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# 실제 파일 위치에 맞게 이 부분만 수정
INPUT = Path("datas/중소기업통합.pkl")

EVENTS = {
    "홍해": "2023-11-19",
    "호르무즈": "2026-02-28",
}

# 사용자 제공 업종 대분류
SECTORS = [
    ("A", "농업, 임업 및 어업", 1, 3),
    ("B", "광업", 5, 8),
    ("C", "제조업", 10, 34),
    ("D", "전기, 가스, 증기 및 공기 조절 공급업", 35, 35),
    ("E", "수도, 하수 및 폐기물 처리, 원료 재생업", 36, 39),
    ("F", "건설업", 41, 42),
    ("G", "도매 및 소매업", 45, 47),
    ("H", "운수 및 창고업", 49, 52),
    ("I", "숙박 및 음식점업", 55, 56),
    ("J", "정보통신업", 58, 63),
    ("K", "금융 및 보험업", 64, 66),
    ("L", "부동산업", 68, 68),
    ("M", "전문, 과학 및 기술 서비스업", 70, 73),
    ("N", "사업시설 관리, 사업 지원 및 임대 서비스업", 74, 76),
    ("O", "공공 행정, 국방 및 사회보장 행정", 84, 84),
    ("P", "교육 서비스업", 85, 85),
    ("Q", "보건업 및 사회복지 서비스업", 86, 87),
    ("R", "예술, 스포츠 및 여가관련 서비스업", 90, 91),
    ("S", "협회 및 단체, 수리 및 기타 개인 서비스업", 94, 96),
    ("T", "가구 내 고용활동 및 자가소비 생산활동", 97, 98),
    ("U", "국제 및 외국기관", 99, 99),
]

LABELS = [f"{a}.{b}" for a, b, _, _ in SECTORS] + ["미분류"]


def sector(v):
    """첨부자료의 3자리 업종코드를 대분류로 변환."""
    s = str(v).strip().replace("[", "").replace("]", "")
    s = s[:-2] if s.endswith(".0") else s

    if not s.isdigit():
        return "미분류"

    # 602 → 60, 107 → 10, 숫자로 읽힌 11 → 011 → 01
    n = int(s.zfill(3)[:2])

    return next(
        (f"{a}.{b}" for a, b, low, high in SECTORS if low <= n <= high),
        "미분류",
    )


def num(s):
    """쉼표·괄호 음수 등을 숫자로 변환. 결측값은 0으로 바꾸지 않음."""
    s = s.astype("string").str.strip().str.replace(",", "", regex=False)
    s = s.str.replace(r"^\((.*)\)$", r"-\1", regex=True)
    return pd.to_numeric(s, errors="coerce")


def prepare(raw):
    raw = raw.copy(deep=False)
    raw.columns = raw.columns.str.strip()

    # 대용량 메모리 절약: 손익 관련 후보 행부터 추림
    code0 = (
        raw.loc[:, raw.columns == "항목코드"]
        .iloc[:, 0].fillna("").astype(str)
    )
    name0 = (
        raw.loc[:, raw.columns == "항목명"]
        .iloc[:, 0].fillna("").astype(str)
    )

    keep = (
        code0.str.contains(
            "Revenue|OperatingIncome|OperatingActivities", regex=True
        )
        | name0.str.contains("매출|수익|영업", regex=True)
    )
    raw = raw.loc[keep].copy()

    # '당기 반기 3개월'과 '당기 반기 3개월 ' 같은 중복 열 병합
    cols = {}
    for c in raw.columns.unique():
        block = raw.loc[:, raw.columns == c]
        v = block.iloc[:, 0]
        for j in range(1, block.shape[1]):
            v = v.combine_first(block.iloc[:, j])
        cols[c] = v

    d = pd.DataFrame(cols)

    code = d["항목코드"].fillna("").astype(str).str.strip()
    name = (
        d["항목명"].fillna("").astype(str)
        .str.replace(r"\s+", "", regex=True)
    )

    rev_std = code.isin([
        "ifrs_Revenue", "ifrs-full_Revenue",
        "dart_Revenue", "dart_OperatingRevenue",
    ])
    op_std = code.isin([
        "dart_OperatingIncomeLoss",
        "ifrs_ProfitLossFromOperatingActivities",
        "ifrs-full_ProfitLossFromOperatingActivities",
    ])

    rev = rev_std | (
        name.isin([
            "매출", "매출액", "수익", "수익(매출액)",
            "매출액(수익)", "영업수익",
        ])
        & ~code.isin(["ifrs_GrossProfit", "ifrs-full_GrossProfit"])
    )
    op = op_std | name.isin([
        "영업이익", "영업이익(손실)", "영업손익", "영업손실",
    ])

    d["항목"] = np.select(
        [rev, op], ["매출액", "영업이익"], default=""
    )
    d["항목우선"] = np.where(rev_std | op_std, 0, 1)
    d["손실표기"] = name.eq("영업손실")
    d = d[d["항목"].ne("")].copy()

    d["기업"] = (
        d["종목코드"].astype("string")
        .str.replace(r"[^0-9]", "", regex=True).str.zfill(6)
    )
    d["기준일"] = pd.to_datetime(d["결산기준일"], errors="coerce")

    # 달력 분기와 회계 분기가 다른 기업은 잘못 비교하지 않도록 제외
    valid = (
        pd.to_numeric(d["결산월"], errors="coerce").eq(12)
        & d["기준일"].dt.is_quarter_end
        & d["기업"].str.fullmatch(r"\d{6}", na=False)
    )
    print(
        "비12월결산·비표준기준일·식별자 누락 제외 항목행:",
        int((~valid).sum()),
    )
    d = d[valid].copy()

    d["기준"] = d["연결여부"].astype(str).str.strip()
    d["표종류"] = d["재무제표구분"].astype(str).str.strip()
    d["업종대분류"] = d["업종"].map(sector)
    d["통화"] = d["통화"].fillna("미상")

    records = []

    for q, tag in [(1, "1분기"), (2, "반기"), (3, "3분기"), (4, "")]:
        x = d[d["기준일"].dt.quarter.eq(q)].copy()

        if x.empty:
            continue

        # 전기 비교공시 값도 활용하되, 해당 연도 당기 공시를 우선
        for side, offset in [("당기", 0), ("전기", 1)]:
            z = x[[
                "기업", "회사명", "업종대분류", "기준",
                "표종류", "통화", "항목", "항목우선",
            ]].copy()

            z["연도"] = x["기준일"].dt.year - offset
            z["분기"] = q
            z["출처우선"] = offset

            def val(c):
                v = (
                    num(x[c]) if c in x
                    else pd.Series(np.nan, index=x.index)
                )
                return v.mask(x["손실표기"], -v.abs())

            z["누적"] = val(
                side if q == 4 else f"{side} {tag} 누적"
            )
            z["단분기"] = (
                val(f"{side} {tag} 3개월") if q < 4 else np.nan
            )

            if q == 1:
                z["단분기"] = z["단분기"].fillna(z["누적"])

            records.append(z)

    if not records:
        raise ValueError("사용 가능한 손익계산서 행이 없습니다.")

    r = pd.concat(records, ignore_index=True)

    keys = [
        "기업", "연도", "분기", "기준", "표종류", "통화", "항목"
    ]

    long = r.melt(
        id_vars=keys + ["출처우선", "항목우선"],
        value_vars=["누적", "단분기"],
        var_name="금액종류",
        value_name="값",
    ).dropna(subset=["값"])

    k = keys + ["금액종류"]
    long["우선"] = long["출처우선"] * 10 + long["항목우선"]
    long = long[
        long["우선"].eq(long.groupby(k)["우선"].transform("min"))
    ]

    # 동일 우선순위에서 금액이 충돌하면 임의 합산하지 않음
    conflict = long.groupby(k)["값"].nunique().gt(1)
    print("상충 금액 제외 건수:", int(conflict.sum()))

    long = long.merge(
        conflict.rename("상충").reset_index(), on=k
    )
    long = long[~long["상충"]].drop_duplicates(k)

    a = long.pivot(
        index=keys, columns="금액종류", values="값"
    ).reset_index()

    for c in ["누적", "단분기"]:
        if c not in a:
            a[c] = np.nan

    # 단분기 값이 없으면 당분기 누적 - 직전 분기 누적
    # 특히 Q4 = 연간 - Q3 누적
    prev = a[keys + ["누적"]].copy()
    prev["분기"] += 1
    prev = prev.rename(columns={"누적": "직전누적"})

    a = a.merge(
        prev, on=keys, how="left", validate="one_to_one"
    )
    a["금액"] = a["단분기"].fillna(a["누적"] - a["직전누적"])
    a.loc[a["분기"].eq(1), "금액"] = a.loc[
        a["분기"].eq(1), "단분기"
    ]

    pkeys = keys[:-1]
    p = a.pivot(
        index=pkeys, columns="항목", values="금액"
    ).reset_index()

    for c in ["매출액", "영업이익"]:
        if c not in p:
            p[c] = np.nan

    meta = (
        r.sort_values("출처우선")
        .drop_duplicates(pkeys)[
            pkeys + ["회사명", "업종대분류"]
        ]
    )
    p = p.merge(
        meta, on=pkeys, how="left", validate="one_to_one"
    )

    p["영업이익률"] = (
        100 * p["영업이익"] / p["매출액"]
    ).where(p["매출액"] > 0)

    p["영업이익률"] = p["영업이익률"].replace(
        [np.inf, -np.inf], np.nan
    )

    return p


def stats(g):
    return pd.Series({
        "기업수": len(g),
        "전년평균_%": g["전년이익률"].mean(),
        "당기평균_%": g["영업이익률"].mean(),
        "전년중앙값_%": g["전년이익률"].median(),
        "당기중앙값_%": g["영업이익률"].median(),
        "변화평균_%p": g["변화_%p"].mean(),
        "변화중앙값_%p": g["변화_%p"].median(),
        "변화25분위_%p": g["변화_%p"].quantile(.25),
        "변화75분위_%p": g["변화_%p"].quantile(.75),
        "악화기업비율_%": 100 * g["변화_%p"].lt(0).mean(),
    })


def compare(p, events):
    tables, groups, gaps, details = [], [], [], []

    pairkeys = [
        "기업", "연도", "분기", "기준", "표종류", "통화"
    ]

    for event, date in events.items():
        start = pd.Period(date, freq="Q")

        for period in [start, start + 1]:
            cur = p[
                (p["연도"] == period.year)
                & (p["분기"] == period.quarter)
            ].copy()

            old = p[
                (p["연도"] == period.year - 1)
                & (p["분기"] == period.quarter)
            ][pairkeys + ["영업이익률"]].copy()

            old["연도"] += 1
            old = old.rename(
                columns={"영업이익률": "전년이익률"}
            )

            z = cur.merge(
                old, on=pairkeys,
                how="inner", validate="one_to_one",
            ).dropna(subset=["영업이익률", "전년이익률"])

            # 같은 기업을 연결·별도로 중복 집계하지 않음
            # 전년과 당년 모두 비교 가능한 별도 우선, 없으면 연결
            z["우선"] = (
                z["기준"].map({"별도": 0, "연결": 1}).fillna(2) * 10
                + z["표종류"].map({
                    "손익계산서": 0, "포괄손익계산서": 1
                }).fillna(2)
            )
            z = (
                z.sort_values(["우선", "통화"])
                .drop_duplicates("기업").copy()
            )

            z["변화_%p"] = z["영업이익률"] - z["전년이익률"]
            z["사건"] = event
            z["분기표시"] = str(period)
            details.append(z)

            rows = []
            for label in LABELS:
                g = z[z["업종대분류"] == label]
                row = stats(g).to_dict()
                row.update({
                    "업종": label,
                    "상태": (
                        "자료 없음" if len(g) == 0
                        else "소표본" if len(g) < 5
                        else "집계 가능"
                    ),
                })
                rows.append(row)

            t = pd.DataFrame(rows)
            t["하락순위_중앙값"] = t["변화중앙값_%p"].rank(
                method="min", ascending=True
            )
            t["사건"] = event
            t["분기"] = str(period)
            tables.append(t)

            # 비제조업은 업종평균의 평균이 아니라 기업들을 합쳐 계산
            comp = {}
            masks = {
                "제조업": z["업종대분류"].eq("C.제조업"),
                "비제조업": ~z["업종대분류"].isin(
                    ["C.제조업", "미분류"]
                ),
            }

            for label, mask in masks.items():
                comp[label] = stats(z[mask])
                groups.append({
                    "사건": event,
                    "분기": str(period),
                    "구분": label,
                    **comp[label].to_dict(),
                })

            gaps.append({
                "사건": event,
                "분기": str(period),
                "평균격차_%p": (
                    comp["제조업"]["변화평균_%p"]
                    - comp["비제조업"]["변화평균_%p"]
                ),
                "중앙값격차_%p": (
                    comp["제조업"]["변화중앙값_%p"]
                    - comp["비제조업"]["변화중앙값_%p"]
                ),
            })

    return (
        pd.concat(tables, ignore_index=True),
        pd.DataFrame(groups),
        pd.DataFrame(gaps),
        pd.concat(details, ignore_index=True),
    )

In [24]:
# ============================================================
# 0. 패키지
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display


# ============================================================
# 1. 기본 설정
# ============================================================

DATA_DIR = Path("datas/손익계산서(PL)")

Q1_FILE = (
    DATA_DIR
    / "2026_1분기보고서_03_포괄손익계산서_연결_20260925.txt"
)

Q2_FILE = (
    DATA_DIR
    / "2026_반기보고서_03_포괄손익계산서_연결_20260925.txt"
)


# ------------------------------------------------------------
# 달력상 2026Q1, 2026Q2를 비교하기 위해
# 12월 결산법인만 사용
# ------------------------------------------------------------

CALENDAR_YEAR_ONLY = True


# ------------------------------------------------------------
# 업종별 최소 기업수
# 5개 미만이면 "소표본"
# ------------------------------------------------------------

MIN_FIRMS = 5


# ------------------------------------------------------------
# IQR 이상치 계수
#
# 정상 범위:
# Q1 - 1.5*IQR ~ Q3 + 1.5*IQR
# ------------------------------------------------------------

IQR_K = 1.5


# ============================================================
# 2. 업종 대분류
# ============================================================

INDUSTRY_GROUPS = [

    ("A.농업, 임업 및 어업", 1, 3),

    ("B.광업", 5, 8),

    ("C.제조업", 10, 34),

    (
        "D.전기, 가스, 증기 및 공기 조절 공급업",
        35,
        35
    ),

    (
        "E.수도, 하수 및 폐기물 처리, 원료 재생업",
        36,
        39
    ),

    ("F.건설업", 41, 42),

    ("G.도매 및 소매업", 45, 47),

    ("H.운수 및 창고업", 49, 52),

    ("I.숙박 및 음식점업", 55, 56),

    ("J.정보통신업", 58, 63),

    ("K.금융 및 보험업", 64, 66),

    ("L.부동산업", 68, 68),

    (
        "M.전문, 과학 및 기술 서비스업",
        70,
        73
    ),

    (
        "N.사업시설 관리, 사업 지원 및 임대 서비스업",
        74,
        76
    ),

    (
        "O.공공 행정, 국방 및 사회보장 행정",
        84,
        84
    ),

    ("P.교육 서비스업", 85, 85),

    (
        "Q.보건업 및 사회복지 서비스업",
        86,
        87
    ),

    (
        "R.예술, 스포츠 및 여가관련 서비스업",
        90,
        91
    ),

    (
        "S.협회 및 단체, 수리 및 기타 개인 서비스업",
        94,
        96
    ),

    (
        "T.가구 내 고용활동 및 자가소비 생산활동",
        97,
        98
    ),

    ("U.국제 및 외국기관", 99, 99),
]


# ============================================================
# 3. 업종코드 → A~U 대분류
# ============================================================

def broad_industry(code):

    try:
        code = int(float(code))

    except:
        return np.nan


    # 예:
    # 272 → 27
    # 602 → 60
    # 763 → 76

    if code >= 100:
        two_digit = code // 10

    else:
        two_digit = code


    for name, start, end in INDUSTRY_GROUPS:

        if start <= two_digit <= end:
            return name


    return np.nan


# ============================================================
# 4. 숫자 변환
# ============================================================

def to_number(series):

    return pd.to_numeric(

        series
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip(),

        errors="coerce"
    )


# ============================================================
# 5. 항목명 정리
# ============================================================

def clean_item_name(series):

    x = (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )


    # 예:
    # I. 매출액
    # Ⅰ. 영업수익
    # 1. 매출액
    #
    # 앞의 번호/기호 제거

    x = x.str.replace(

        r"^[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX0-9\.\-\s]+",

        "",

        regex=True
    )


    return x.str.strip()


# ============================================================
# 6. DART TXT 읽기
# ============================================================

def read_dart_file(path):

    try:

        df = pd.read_csv(

            path,

            sep="\t",

            encoding="cp949",

            dtype=str,

            low_memory=False
        )

    except UnicodeDecodeError:

        df = pd.read_csv(

            path,

            sep="\t",

            encoding="utf-8-sig",

            dtype=str,

            low_memory=False
        )


    # 컬럼명 앞뒤 공백 제거

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )


    return df


# ============================================================
# 7. 영업이익 추출
# ============================================================

def extract_operating_income(
    df,
    current_col,
    previous_col
):

    x = df.copy()


    x["항목코드"] = (
        x["항목코드"]
        .fillna("")
        .astype(str)
        .str.strip()
    )


    x["항목명정리"] = (
        clean_item_name(
            x["항목명"]
        )
    )


    # --------------------------------------------------------
    # 1순위: DART 표준 영업이익 코드
    # --------------------------------------------------------

    standard = x[
        x["항목코드"]
        == "dart_OperatingIncomeLoss"
    ].copy()


    standard["선택우선순위"] = 1


    standard_firms = set(
        standard["기업키"]
    )


    # --------------------------------------------------------
    # 2순위:
    # 표준코드 없는 기업은 항목명으로 찾기
    # --------------------------------------------------------

    fallback = x[
        ~x["기업키"].isin(
            standard_firms
        )
    ].copy()


    operating_names = {

        "영업이익",

        "영업이익(손실)",

        "영업손익",

        "영업이익손실"
    }


    fallback = fallback[
        fallback["항목명정리"]
        .isin(operating_names)
    ].copy()


    fallback["선택우선순위"] = 2


    result = pd.concat(

        [
            standard,
            fallback
        ],

        ignore_index=True
    )


    # 기업별 1개만 선택

    result = (

        result

        .sort_values(
            [
                "기업키",
                "선택우선순위"
            ]
        )

        .drop_duplicates(
            subset=["기업키"],
            keep="first"
        )
    )


    result = result[
        [
            "기업키",
            current_col,
            previous_col
        ]
    ].copy()


    result = result.rename(

        columns={

            current_col:
                "영업이익_당기",

            previous_col:
                "영업이익_전년"
        }
    )


    return result


# ============================================================
# 8. 매출액 추출
# ============================================================

def extract_revenue(
    df,
    current_col,
    previous_col
):

    x = df.copy()


    x["항목코드"] = (
        x["항목코드"]
        .fillna("")
        .astype(str)
        .str.strip()
    )


    x["항목명정리"] = (
        clean_item_name(
            x["항목명"]
        )
    )


    # --------------------------------------------------------
    # 1순위: IFRS 표준 Revenue
    # --------------------------------------------------------

    standard = x[
        x["항목코드"]
        == "ifrs-full_Revenue"
    ].copy()


    standard["선택우선순위"] = 1


    standard_firms = set(
        standard["기업키"]
    )


    # --------------------------------------------------------
    # 2순위:
    # 자체 XBRL 항목을 쓰는 기업
    # --------------------------------------------------------

    fallback = x[
        ~x["기업키"].isin(
            standard_firms
        )
    ].copy()


    revenue_names = {

        "매출액",

        "매출",

        "수익(매출액)",

        "영업수익",

        "매출액 및 기타영업수익",

        "매출액 및 지분법이익",

        "매출 및 지분법손익"
    }


    fallback = fallback[
        fallback["항목명정리"]
        .isin(revenue_names)
    ].copy()


    fallback["선택우선순위"] = 2


    result = pd.concat(

        [
            standard,
            fallback
        ],

        ignore_index=True
    )


    result = (

        result

        .sort_values(
            [
                "기업키",
                "선택우선순위"
            ]
        )

        .drop_duplicates(
            subset=["기업키"],
            keep="first"
        )
    )


    result = result[
        [
            "기업키",
            current_col,
            previous_col
        ]
    ].copy()


    result = result.rename(

        columns={

            current_col:
                "매출_당기",

            previous_col:
                "매출_전년"
        }
    )


    return result


# ============================================================
# 9. 기업별 영업이익률 계산
# ============================================================

def make_quarter_data(
    path,
    current_col,
    previous_col
):

    df = read_dart_file(path)


    current_col = current_col.strip()
    previous_col = previous_col.strip()


    # ========================================================
    # 9-1. 12월 결산법인만 사용
    # ========================================================

    if CALENDAR_YEAR_ONLY:

        df["결산월"] = (

            df["결산월"]

            .fillna("")

            .astype(str)

            .str.strip()

            .str.zfill(2)
        )


        df = df[
            df["결산월"] == "12"
        ].copy()


    # ========================================================
    # 9-2. 기업키 만들기
    # ========================================================

    code = (

        df["종목코드"]

        .fillna("")

        .astype(str)

        .str.strip()
    )


    invalid_code = (

        code
        .str.lower()
        .isin(
            [
                "",
                "nan",
                "null",
                "[null]",
                "none"
            ]
        )
    )


    # 종목코드가 있으면 종목코드
    # 없으면 회사명 사용

    df["기업키"] = np.where(

        invalid_code,

        df["회사명"]
        .fillna("")
        .astype(str)
        .str.strip(),

        code
    )


    # ========================================================
    # 9-3. 기업 기본 정보
    # ========================================================

    meta_cols = [

        "기업키",

        "종목코드",

        "회사명",

        "시장구분",

        "업종",

        "업종명",

        "결산월",

        "결산기준일"
    ]


    meta = (

        df[meta_cols]

        .drop_duplicates(
            subset=["기업키"]
        )

        .copy()
    )


    # ========================================================
    # 9-4. 영업이익
    # ========================================================

    operating = extract_operating_income(

        df,

        current_col,

        previous_col
    )


    # ========================================================
    # 9-5. 매출
    # ========================================================

    revenue = extract_revenue(

        df,

        current_col,

        previous_col
    )


    # ========================================================
    # 9-6. 합치기
    # ========================================================

    data = (

        meta

        .merge(
            operating,
            on="기업키",
            how="inner"
        )

        .merge(
            revenue,
            on="기업키",
            how="inner"
        )
    )


    # ========================================================
    # 9-7. 숫자 변환
    # ========================================================

    numeric_cols = [

        "영업이익_당기",

        "영업이익_전년",

        "매출_당기",

        "매출_전년"
    ]


    for col in numeric_cols:

        data[col] = (
            to_number(
                data[col]
            )
        )


    data = data.dropna(
        subset=numeric_cols
    )


    # ========================================================
    # 9-8. 매출 0 이하 제외
    # ========================================================

    data = data[

        (data["매출_당기"] > 0)

        &

        (data["매출_전년"] > 0)

    ].copy()


    # ========================================================
    # ★ 9-9. 기업별 영업이익률
    #
    # 반드시 여기서 먼저 기업별 계산
    # ========================================================

    data["전년영업이익률_%"] = (

        data["영업이익_전년"]

        /

        data["매출_전년"]

        * 100
    )


    data["당기영업이익률_%"] = (

        data["영업이익_당기"]

        /

        data["매출_당기"]

        * 100
    )


    # 기업별 변화

    data["변화_%p"] = (

        data["당기영업이익률_%"]

        -

        data["전년영업이익률_%"]
    )


    # ========================================================
    # 9-10. 업종 대분류
    # ========================================================

    data["대분류"] = (

        data["업종"]

        .apply(
            broad_industry
        )
    )


    data = data.dropna(
        subset=["대분류"]
    )


    # ========================================================
    # 9-11. 제조업 / 비제조업
    # ========================================================

    data["제조업구분"] = np.where(

        data["대분류"]
        == "C.제조업",

        "제조업",

        "비제조업"
    )


    return data


# ============================================================
# 10. ★ IQR 방식 이상치 제거
#
# 정상범위:
#
# Q1 - 1.5*IQR
# ~
# Q3 + 1.5*IQR
#
# 전년 또는 당기 중 하나라도 이상치면
# 해당 기업을 비교표본에서 제외
# ============================================================

def remove_outliers_iqr(data):

    x = data.copy()


    before = len(x)


    # ========================================================
    # 10-1. 전년도 영업이익률
    # ========================================================

    prev_q1 = (
        x["전년영업이익률_%"]
        .quantile(0.25)
    )


    prev_median = (
        x["전년영업이익률_%"]
        .median()
    )


    prev_q3 = (
        x["전년영업이익률_%"]
        .quantile(0.75)
    )


    prev_iqr = (
        prev_q3
        - prev_q1
    )


    prev_lower = (
        prev_q1
        - IQR_K * prev_iqr
    )


    prev_upper = (
        prev_q3
        + IQR_K * prev_iqr
    )


    # ========================================================
    # 10-2. 당기 영업이익률
    # ========================================================

    curr_q1 = (
        x["당기영업이익률_%"]
        .quantile(0.25)
    )


    curr_median = (
        x["당기영업이익률_%"]
        .median()
    )


    curr_q3 = (
        x["당기영업이익률_%"]
        .quantile(0.75)
    )


    curr_iqr = (
        curr_q3
        - curr_q1
    )


    curr_lower = (
        curr_q1
        - IQR_K * curr_iqr
    )


    curr_upper = (
        curr_q3
        + IQR_K * curr_iqr
    )


    # ========================================================
    # 10-3. 이상치 여부
    # ========================================================

    prev_normal = (

        x["전년영업이익률_%"]

        .between(

            prev_lower,

            prev_upper,

            inclusive="both"
        )
    )


    curr_normal = (

        x["당기영업이익률_%"]

        .between(

            curr_lower,

            curr_upper,

            inclusive="both"
        )
    )


    # 전년과 당기 모두 정상인 기업만 남김

    normal_mask = (
        prev_normal
        &
        curr_normal
    )


    removed_data = (
        x[
            ~normal_mask
        ]
        .copy()
    )


    x = (
        x[
            normal_mask
        ]
        .copy()
    )


    # ========================================================
    # 10-4. 제거 현황
    # ========================================================

    after = len(x)

    removed = (
        before
        - after
    )


    if before > 0:

        removed_rate = (
            removed
            / before
            * 100
        )

    else:

        removed_rate = 0


    # ========================================================
    # 10-5. 변화량 재계산
    # ========================================================

    x["변화_%p"] = (

        x["당기영업이익률_%"]

        -

        x["전년영업이익률_%"]
    )


    x["악화여부"] = (

        x["변화_%p"]
        < 0
    )


    # ========================================================
    # 10-6. IQR 정보 저장
    # ========================================================

    info = {

        "원본기업수":
            before,

        "분석기업수":
            after,

        "제거기업수":
            removed,

        "제거율":
            removed_rate,


        # 전년

        "전년_Q1":
            prev_q1,

        "전년_중앙값":
            prev_median,

        "전년_Q3":
            prev_q3,

        "전년_IQR":
            prev_iqr,

        "전년하한":
            prev_lower,

        "전년상한":
            prev_upper,


        # 당기

        "당기_Q1":
            curr_q1,

        "당기_중앙값":
            curr_median,

        "당기_Q3":
            curr_q3,

        "당기_IQR":
            curr_iqr,

        "당기하한":
            curr_lower,

        "당기상한":
            curr_upper
    }


    return (
        x,
        removed_data,
        info
    )


# ============================================================
# 11. 제조업/업종별 집계
#
# ★ 모든 기업 가중치 = 1
#
# 예:
# 기업 A = 4%
# 기업 B = 5%
# 기업 C = 6%
#
# 평균 = 5%
# ============================================================

def summary_table(
    data,
    group_col
):

    # 혹시 모를 중복 제거
    # 기업당 정확히 한 행

    firm = (

        data

        .drop_duplicates(
            subset=[
                "기업키",
                group_col
            ]
        )

        .copy()
    )


    result = (

        firm

        .groupby(
            group_col
        )

        .agg(

            기업수=(
                "기업키",
                "nunique"
            ),


            # ----------------------------------------------
            # 각 기업의 영업이익률 단순 평균
            # ----------------------------------------------

            전년평균_=(
                "전년영업이익률_%",
                "mean"
            ),

            당기평균_=(
                "당기영업이익률_%",
                "mean"
            ),


            # ----------------------------------------------
            # 기업별 변화량의 단순 평균
            # ----------------------------------------------

            변화평균_=(
                "변화_%p",
                "mean"
            ),


            변화중앙값_=(
                "변화_%p",
                "median"
            ),


            악화기업비율_=(
                "악화여부",
                "mean"
            )
        )

        .reset_index()
    )


    # 비율 → %

    result[
        "악화기업비율_"
    ] *= 100


    result = result.rename(

        columns={

            "전년평균_":
                "전년평균_%",

            "당기평균_":
                "당기평균_%",

            "변화평균_":
                "변화평균_%p",

            "변화중앙값_":
                "변화중앙값_%p",

            "악화기업비율_":
                "악화기업비율_%"
        }
    )


    # 반올림

    numeric_cols = [

        "전년평균_%",

        "당기평균_%",

        "변화평균_%p",

        "변화중앙값_%p",

        "악화기업비율_%"
    ]


    result[
        numeric_cols
    ] = (

        result[
            numeric_cols
        ]

        .round(2)
    )


    return result


# ============================================================
# 12. 표 색상
# ============================================================

def change_color(v):

    if pd.isna(v):
        return ""


    # 하락 = 빨강

    if v < 0:

        return (
            "background-color: #fdeaea;"
            "color: #d32f2f;"
            "font-weight: bold;"
        )


    # 상승 = 초록

    elif v > 0:

        return (
            "background-color: #e7f6ec;"
            "color: #087b35;"
            "font-weight: bold;"
        )


    return ""


def sample_color(v):

    if v == "소표본":

        return (
            "background-color: #fff3cd;"
            "color: #8a6500;"
            "font-weight: bold;"
        )


    return (
        "background-color: #e7f0ff;"
        "color: #2457a6;"
    )


# ============================================================
# 13. 예쁜 표
# ============================================================

def pretty_table(
    df,
    title,
    industry=False
):

    x = df.copy()


    # ========================================================
    # 순위 추가
    # ========================================================

    if industry:

        x = (
            x
            .reset_index(
                drop=True
            )
        )


        x.insert(

            0,

            "순위",

            np.arange(
                1,
                len(x) + 1
            )
        )


    # ========================================================
    # 컬럼명 변경
    # ========================================================

    rename_map = {

        "제조업구분":
            "구분",

        "대분류":
            "업종",

        "기업수":
            "기업 수",

        "전년평균_%":
            "전년 영업이익률",

        "당기평균_%":
            "당기 영업이익률",

        "변화평균_%p":
            "평균 변화",

        "변화중앙값_%p":
            "중앙값 변화",

        "악화기업비율_%":
            "악화 기업 비율",

        "상태":
            "표본 상태"
    }


    x = x.rename(
        columns=rename_map
    )


    # ========================================================
    # 숫자 표시 형식
    # ========================================================

    fmt = {}


    if "기업 수" in x.columns:

        fmt[
            "기업 수"
        ] = "{:,.0f}"


    if "전년 영업이익률" in x.columns:

        fmt[
            "전년 영업이익률"
        ] = "{:.2f}%"


    if "당기 영업이익률" in x.columns:

        fmt[
            "당기 영업이익률"
        ] = "{:.2f}%"


    if "평균 변화" in x.columns:

        fmt[
            "평균 변화"
        ] = "{:+.2f}%p"


    if "중앙값 변화" in x.columns:

        fmt[
            "중앙값 변화"
        ] = "{:+.2f}%p"


    if "악화 기업 비율" in x.columns:

        fmt[
            "악화 기업 비율"
        ] = "{:.1f}%"


    # ========================================================
    # 스타일
    # ========================================================

    styler = (

        x.style

        .hide(axis="index")

        .format(
            fmt,
            na_rep="-"
        )

        .set_caption(
            title
        )

        .set_table_styles(
            [

                {
                    "selector":
                        "caption",

                    "props": [

                        (
                            "caption-side",
                            "top"
                        ),

                        (
                            "font-size",
                            "20px"
                        ),

                        (
                            "font-weight",
                            "700"
                        ),

                        (
                            "text-align",
                            "left"
                        ),

                        (
                            "padding",
                            "14px 4px"
                        )
                    ]
                },


                {
                    "selector":
                        "thead th",

                    "props": [

                        (
                            "background-color",
                            "#343a40"
                        ),

                        (
                            "color",
                            "white"
                        ),

                        (
                            "font-weight",
                            "700"
                        ),

                        (
                            "text-align",
                            "center"
                        ),

                        (
                            "padding",
                            "10px 12px"
                        )
                    ]
                },


                {
                    "selector":
                        "tbody td",

                    "props": [

                        (
                            "padding",
                            "10px 12px"
                        ),

                        (
                            "text-align",
                            "center"
                        ),

                        (
                            "border-bottom",
                            "1px solid #aaaaaa"
                        )
                    ]
                },


                {
                    "selector":
                        "table",

                    "props": [

                        (
                            "border-collapse",
                            "collapse"
                        ),

                        (
                            "font-size",
                            "15px"
                        ),

                        (
                            "width",
                            "100%"
                        )
                    ]
                }
            ]
        )
    )


    # ========================================================
    # 변화량 색상
    # ========================================================

    change_cols = [

        c

        for c in [

            "평균 변화",

            "중앙값 변화"

        ]

        if c in x.columns
    ]


    if change_cols:

        styler = styler.applymap(

            change_color,

            subset=change_cols
        )


    # ========================================================
    # 표본상태 색상
    # ========================================================

    if "표본 상태" in x.columns:

        styler = styler.applymap(

            sample_color,

            subset=[
                "표본 상태"
            ]
        )


    # 업종명 좌측 정렬

    if "업종" in x.columns:

        styler = styler.set_properties(

            subset=[
                "업종"
            ],

            **{

                "text-align":
                    "left",

                "font-weight":
                    "600"
            }
        )


    # 제조업/비제조업 강조

    if "구분" in x.columns:

        styler = styler.set_properties(

            subset=[
                "구분"
            ],

            **{
                "font-weight":
                    "700"
            }
        )


    return styler


# ============================================================
# 14. 한 기간 분석
# ============================================================

def analyze_period(
    path,
    current_col,
    previous_col,
    title
):

    # ========================================================
    # ① 기업별 영업이익률 계산
    # ========================================================

    raw = make_quarter_data(

        path,

        current_col,

        previous_col
    )


    # ========================================================
    # ② IQR 이상치 제거
    # ========================================================

    (
        data,
        removed_data,
        outlier_info

    ) = remove_outliers_iqr(
        raw
    )


    # ========================================================
    # ③ 이상치 제거 정보 출력
    # ========================================================

    print(
        "\n"
        + "=" * 80
    )

    print(title)

    print(
        "=" * 80
    )


    print(
        "\n[IQR 이상치 제거]"
    )


    print(
        "\n기준:"
    )

    print(
        "Q1 - 1.5×IQR "
        "~ "
        "Q3 + 1.5×IQR"
    )


    print(
        f"\n원본 기업 수 : "
        f"{outlier_info['원본기업수']:,}"
    )


    print(
        f"분석 기업 수 : "
        f"{outlier_info['분석기업수']:,}"
    )


    print(
        f"제거 기업 수 : "
        f"{outlier_info['제거기업수']:,}"
    )


    print(
        f"제거율 : "
        f"{outlier_info['제거율']:.2f}%"
    )


    # ========================================================
    # 전년 IQR 정보
    # ========================================================

    print(
        "\n[전년 영업이익률 분포]"
    )


    print(

        f"Q1 = "
        f"{outlier_info['전년_Q1']:.2f}%"
    )


    print(

        f"중앙값 = "
        f"{outlier_info['전년_중앙값']:.2f}%"
    )


    print(

        f"Q3 = "
        f"{outlier_info['전년_Q3']:.2f}%"
    )


    print(

        f"IQR = "
        f"{outlier_info['전년_IQR']:.2f}%p"
    )


    print(

        f"정상 범위 = "

        f"{outlier_info['전년하한']:.2f}%"

        " ~ "

        f"{outlier_info['전년상한']:.2f}%"
    )


    # ========================================================
    # 당기 IQR 정보
    # ========================================================

    print(
        "\n[당기 영업이익률 분포]"
    )


    print(

        f"Q1 = "
        f"{outlier_info['당기_Q1']:.2f}%"
    )


    print(

        f"중앙값 = "
        f"{outlier_info['당기_중앙값']:.2f}%"
    )


    print(

        f"Q3 = "
        f"{outlier_info['당기_Q3']:.2f}%"
    )


    print(

        f"IQR = "
        f"{outlier_info['당기_IQR']:.2f}%p"
    )


    print(

        f"정상 범위 = "

        f"{outlier_info['당기하한']:.2f}%"

        " ~ "

        f"{outlier_info['당기상한']:.2f}%"
    )


    # ========================================================
    # ④ 제조업 vs 비제조업
    # ========================================================

    manufacturing = summary_table(

        data,

        "제조업구분"
    )


    # 제조업 먼저 출력

    order = {

        "제조업": 0,

        "비제조업": 1
    }


    manufacturing["_order"] = (

        manufacturing[
            "제조업구분"
        ]

        .map(order)
    )


    manufacturing = (

        manufacturing

        .sort_values(
            "_order"
        )

        .drop(
            columns="_order"
        )

        .reset_index(
            drop=True
        )
    )


    # ========================================================
    # ⑤ 업종별
    # ========================================================

    industry = summary_table(

        data,

        "대분류"
    )


    # 표본상태

    industry["상태"] = np.where(

        industry["기업수"]
        >= MIN_FIRMS,

        "집계 가능",

        "소표본"
    )


    # ========================================================
    # ★ 영업이익률이 많이 낮아진 순서
    # ========================================================

    industry = (

        industry

        .sort_values(

            [
                "변화평균_%p",
                "기업수"
            ],

            ascending=[
                True,
                False
            ]
        )

        .reset_index(
            drop=True
        )
    )


    # ========================================================
    # ⑥ 예쁜 표 출력
    # ========================================================

    display(

        pretty_table(

            manufacturing,

            title=(
                f"📊 {title} | "
                f"제조업 vs 비제조업"
            )
        )
    )


    display(

        pretty_table(

            industry,

            title=(
                f"📉 {title} | "
                f"업종별 영업이익률 변화"
            ),

            industry=True
        )
    )


    # ========================================================
    # 결과 반환
    # ========================================================

    return {

        "원본기업":
            raw,

        "분석기업":
            data,

        "제거기업":
            removed_data,

        "제조업비교":
            manufacturing,

        "업종별":
            industry,

        "IQR정보":
            outlier_info
    }


# ============================================================
# 15. 호르무즈
# 2026 Q1 vs 2025 Q1
# ============================================================

q1_result = analyze_period(

    Q1_FILE,

    current_col=
        "당기 1분기 3개월",

    previous_col=
        "전기 1분기 3개월",

    title=
        "호르무즈 | 2026Q1 vs 2025Q1"
)


# ============================================================
# 16. 호르무즈
# 2026 Q2 vs 2025 Q2
# ============================================================

q2_result = analyze_period(

    Q2_FILE,

    current_col=
        "당기 반기 3개월",

    previous_col=
        "전기 반기 3개월",

    title=
        "호르무즈 | 2026Q2 vs 2025Q2"
)


호르무즈 | 2026Q1 vs 2025Q1

[IQR 이상치 제거]

기준:
Q1 - 1.5×IQR ~ Q3 + 1.5×IQR

원본 기업 수 : 2,005
분석 기업 수 : 1,623
제거 기업 수 : 382
제거율 : 19.05%

[전년 영업이익률 분포]
Q1 = -9.01%
중앙값 = 2.07%
Q3 = 7.28%
IQR = 16.30%p
정상 범위 = -33.46% ~ 31.73%

[당기 영업이익률 분포]
Q1 = -7.64%
중앙값 = 2.54%
Q3 = 8.26%
IQR = 15.89%p
정상 범위 = -31.48% ~ 32.10%


구분,기업 수,전년 영업이익률,당기 영업이익률,평균 변화,중앙값 변화,악화 기업 비율
제조업,"1,065",2.78%,3.21%,+0.43%p,+0.20%p,47.4%
비제조업,558,2.17%,3.35%,+1.18%p,+0.80%p,40.9%


순위,업종,기업 수,전년 영업이익률,당기 영업이익률,평균 변화,중앙값 변화,악화 기업 비율,표본 상태
1,I.숙박 및 음식점업,6,10.83%,8.72%,-2.11%p,+0.23%p,50.0%,집계 가능
2,"E.수도, 하수 및 폐기물 처리, 원료 재생업",3,12.53%,10.50%,-2.03%p,-1.81%p,100.0%,소표본
3,L.부동산업,7,0.31%,-0.16%,-0.46%p,+0.07%p,42.9%,집계 가능
4,G.도매 및 소매업,112,2.01%,2.31%,+0.30%p,+0.28%p,43.8%,집계 가능
5,"N.사업시설 관리, 사업 지원 및 임대 서비스업",20,8.18%,8.53%,+0.35%p,+0.49%p,40.0%,집계 가능
6,C.제조업,"1,065",2.78%,3.21%,+0.43%p,+0.20%p,47.4%,집계 가능
7,"M.전문, 과학 및 기술 서비스업",61,-0.69%,0.19%,+0.89%p,+0.72%p,45.9%,집계 가능
8,"S.협회 및 단체, 수리 및 기타 개인 서비스업",1,17.98%,18.87%,+0.89%p,+0.89%p,0.0%,소표본
9,H.운수 및 창고업,20,4.80%,5.89%,+1.10%p,-0.15%p,50.0%,집계 가능
10,J.정보통신업,194,0.27%,1.52%,+1.25%p,+0.94%p,44.3%,집계 가능



호르무즈 | 2026Q2 vs 2025Q2

[IQR 이상치 제거]

기준:
Q1 - 1.5×IQR ~ Q3 + 1.5×IQR

원본 기업 수 : 2,005
분석 기업 수 : 1,602
제거 기업 수 : 403
제거율 : 20.10%

[전년 영업이익률 분포]
Q1 = -5.05%
중앙값 = 3.03%
Q3 = 9.04%
IQR = 14.09%p
정상 범위 = -26.19% ~ 30.17%

[당기 영업이익률 분포]
Q1 = -4.35%
중앙값 = 3.86%
Q3 = 10.35%
IQR = 14.69%p
정상 범위 = -26.39% ~ 32.39%


구분,기업 수,전년 영업이익률,당기 영업이익률,평균 변화,중앙값 변화,악화 기업 비율
제조업,"1,054",3.95%,5.26%,+1.31%p,+0.91%p,41.6%
비제조업,548,4.40%,5.43%,+1.03%p,+0.80%p,42.3%


순위,업종,기업 수,전년 영업이익률,당기 영업이익률,평균 변화,중앙값 변화,악화 기업 비율,표본 상태
1,"R.예술, 스포츠 및 여가관련 서비스업",8,13.48%,7.45%,-6.03%p,-4.11%p,75.0%,집계 가능
2,I.숙박 및 음식점업,6,6.15%,2.06%,-4.09%p,-0.03%p,50.0%,집계 가능
3,"N.사업시설 관리, 사업 지원 및 임대 서비스업",18,6.69%,3.89%,-2.81%p,-0.66%p,61.1%,집계 가능
4,"S.협회 및 단체, 수리 및 기타 개인 서비스업",1,19.28%,17.56%,-1.72%p,-1.72%p,100.0%,소표본
5,P.교육 서비스업,8,7.56%,6.82%,-0.74%p,+0.78%p,37.5%,집계 가능
6,"E.수도, 하수 및 폐기물 처리, 원료 재생업",3,6.27%,6.61%,+0.34%p,-0.04%p,66.7%,소표본
7,"M.전문, 과학 및 기술 서비스업",60,3.24%,3.73%,+0.49%p,+1.41%p,43.3%,집계 가능
8,F.건설업,40,4.84%,5.42%,+0.58%p,+0.36%p,42.5%,집계 가능
9,C.제조업,"1,054",3.95%,5.26%,+1.31%p,+0.91%p,41.6%,집계 가능
10,J.정보통신업,194,3.85%,5.24%,+1.39%p,+0.96%p,43.8%,집계 가능


In [26]:
# ============================================================
# 0. 패키지
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display


# ============================================================
# 1. 기본 설정
# ============================================================

DATA_DIR = Path("datas/손익계산서(PL)")

# ------------------------------------------------------------
# 홍해 2023Q4 vs 2022Q4
# ------------------------------------------------------------

ANNUAL_2023_FILE = (
    DATA_DIR
    / "2023_사업보고서_03_포괄손익계산서_연결_20260814.txt"
)

# ★ 이 파일은 추가로 필요
Q3_2023_FILE = (
    DATA_DIR
    / "2023_3분기보고서_03_포괄손익계산서_연결_20260814.txt"
)


# ------------------------------------------------------------
# 홍해 2024Q1 vs 2023Q1
# ------------------------------------------------------------

Q1_2024_FILE = (
    DATA_DIR
    / "2024_1분기보고서_03_포괄손익계산서_연결_20260814.txt"
)


# ------------------------------------------------------------
# 설정
# ------------------------------------------------------------

CALENDAR_YEAR_ONLY = True

MIN_FIRMS = 5

# Tukey IQR 방식
IQR_K = 1.5


# ============================================================
# 2. 업종 대분류
# ============================================================

INDUSTRY_GROUPS = [

    ("A.농업, 임업 및 어업", 1, 3),

    ("B.광업", 5, 8),

    ("C.제조업", 10, 34),

    (
        "D.전기, 가스, 증기 및 공기 조절 공급업",
        35, 35
    ),

    (
        "E.수도, 하수 및 폐기물 처리, 원료 재생업",
        36, 39
    ),

    ("F.건설업", 41, 42),

    ("G.도매 및 소매업", 45, 47),

    ("H.운수 및 창고업", 49, 52),

    ("I.숙박 및 음식점업", 55, 56),

    ("J.정보통신업", 58, 63),

    ("K.금융 및 보험업", 64, 66),

    ("L.부동산업", 68, 68),

    (
        "M.전문, 과학 및 기술 서비스업",
        70, 73
    ),

    (
        "N.사업시설 관리, 사업 지원 및 임대 서비스업",
        74, 76
    ),

    (
        "O.공공 행정, 국방 및 사회보장 행정",
        84, 84
    ),

    ("P.교육 서비스업", 85, 85),

    (
        "Q.보건업 및 사회복지 서비스업",
        86, 87
    ),

    (
        "R.예술, 스포츠 및 여가관련 서비스업",
        90, 91
    ),

    (
        "S.협회 및 단체, 수리 및 기타 개인 서비스업",
        94, 96
    ),

    (
        "T.가구 내 고용활동 및 자가소비 생산활동",
        97, 98
    ),

    ("U.국제 및 외국기관", 99, 99),
]


# ============================================================
# 3. 업종코드 → 대분류
# ============================================================

def broad_industry(code):

    try:
        code = int(float(code))

    except:
        return np.nan

    # 272 → 27
    # 602 → 60
    # 763 → 76

    if code >= 100:
        two_digit = code // 10

    else:
        two_digit = code

    for name, start, end in INDUSTRY_GROUPS:

        if start <= two_digit <= end:
            return name

    return np.nan


# ============================================================
# 4. 숫자 변환
# ============================================================

def to_number(series):

    return pd.to_numeric(

        series
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip(),

        errors="coerce"
    )


# ============================================================
# 5. 항목명 정리
# ============================================================

def clean_item_name(series):

    x = (
        series
        .fillna("")
        .astype(str)
        .str.strip()
    )

    x = x.str.replace(
        r"^[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX0-9\.\-\s]+",
        "",
        regex=True
    )

    return x.str.strip()


# ============================================================
# 6. TXT 읽기
# ============================================================

def read_dart_file(path):

    try:

        df = pd.read_csv(
            path,
            sep="\t",
            encoding="cp949",
            dtype=str,
            low_memory=False
        )

    except UnicodeDecodeError:

        df = pd.read_csv(
            path,
            sep="\t",
            encoding="utf-8-sig",
            dtype=str,
            low_memory=False
        )

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
    )

    return df


# ============================================================
# 7. 기업키 생성
# ============================================================

def prepare_base(df):

    x = df.copy()

    # --------------------------------------------------------
    # 12월 결산법인만 사용
    # --------------------------------------------------------

    if CALENDAR_YEAR_ONLY:

        x["결산월"] = (
            x["결산월"]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.zfill(2)
        )

        x = x[
            x["결산월"] == "12"
        ].copy()


    # --------------------------------------------------------
    # 기업키
    # --------------------------------------------------------

    code = (
        x["종목코드"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    invalid = (
        code
        .str.lower()
        .isin([
            "",
            "nan",
            "null",
            "[null]",
            "none"
        ])
    )

    x["기업키"] = np.where(
        invalid,
        x["회사명"]
        .fillna("")
        .astype(str)
        .str.strip(),
        code
    )

    x["항목코드"] = (
        x["항목코드"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    x["항목명정리"] = (
        clean_item_name(
            x["항목명"]
        )
    )

    return x


# ============================================================
# 8. 특정 재무항목 추출
#
# kind = "operating" 또는 "revenue"
# ============================================================

def extract_item(
    df,
    kind,
    value_cols
):

    x = df.copy()

    if kind == "operating":

        standard_code = (
            "dart_OperatingIncomeLoss"
        )

        fallback_names = {
            "영업이익",
            "영업이익(손실)",
            "영업손익",
            "영업이익손실"
        }

    elif kind == "revenue":

        standard_code = (
            "ifrs-full_Revenue"
        )

        fallback_names = {
            "매출액",
            "매출",
            "수익(매출액)",
            "영업수익",
            "매출액 및 기타영업수익",
            "매출액 및 지분법이익",
            "매출 및 지분법손익"
        }

    else:
        raise ValueError(
            "kind는 operating 또는 revenue"
        )


    # --------------------------------------------------------
    # 표준 코드
    # --------------------------------------------------------

    standard = x[
        x["항목코드"]
        == standard_code
    ].copy()

    standard["선택우선순위"] = 1

    standard_firms = set(
        standard["기업키"]
    )


    # --------------------------------------------------------
    # 항목명 fallback
    # --------------------------------------------------------

    fallback = x[
        ~x["기업키"]
        .isin(standard_firms)
    ].copy()

    fallback = fallback[
        fallback["항목명정리"]
        .isin(fallback_names)
    ].copy()

    fallback["선택우선순위"] = 2


    result = pd.concat(
        [standard, fallback],
        ignore_index=True
    )

    result = (
        result
        .sort_values(
            [
                "기업키",
                "선택우선순위"
            ]
        )
        .drop_duplicates(
            subset=["기업키"],
            keep="first"
        )
    )


    return result[
        ["기업키"] + value_cols
    ].copy()


# ============================================================
# 9. 기업 기본정보
# ============================================================

def get_meta(df):

    cols = [
        "기업키",
        "종목코드",
        "회사명",
        "시장구분",
        "업종",
        "업종명",
        "결산월",
        "결산기준일"
    ]

    return (
        df[cols]
        .drop_duplicates(
            subset=["기업키"]
        )
        .copy()
    )


# ============================================================
# 10. ★ 2024Q1 vs 2023Q1
#
# 2024 1분기보고서 안의
#
# 당기 1분기 3개월
# 전기 1분기 3개월
#
# 직접 비교
# ============================================================

def make_q1_data(path):

    df = read_dart_file(path)

    # 공백 제거
    df.columns = (
        df.columns
        .str.strip()
    )

    df = prepare_base(df)

    current_col = (
        "당기 1분기 3개월"
    )

    previous_col = (
        "전기 1분기 3개월"
    )


    # --------------------------------------------------------
    # 영업이익
    # --------------------------------------------------------

    operating = extract_item(
        df,
        "operating",
        [
            current_col,
            previous_col
        ]
    )

    operating = operating.rename(
        columns={
            current_col:
                "영업이익_당기",

            previous_col:
                "영업이익_전년"
        }
    )


    # --------------------------------------------------------
    # 매출
    # --------------------------------------------------------

    revenue = extract_item(
        df,
        "revenue",
        [
            current_col,
            previous_col
        ]
    )

    revenue = revenue.rename(
        columns={
            current_col:
                "매출_당기",

            previous_col:
                "매출_전년"
        }
    )


    # --------------------------------------------------------
    # 합치기
    # --------------------------------------------------------

    meta = get_meta(df)

    data = (
        meta
        .merge(
            operating,
            on="기업키",
            how="inner"
        )
        .merge(
            revenue,
            on="기업키",
            how="inner"
        )
    )


    return finalize_firm_data(data)


# ============================================================
# 11. ★ 2023Q4 vs 2022Q4
#
# Q4 =
# 사업보고서 연간값 - 3분기 누적값
# ============================================================

def make_q4_data(
    annual_path,
    q3_path
):

    # ========================================================
    # 11-1. 사업보고서
    # ========================================================

    annual = read_dart_file(
        annual_path
    )

    annual = prepare_base(
        annual
    )


    # 사업보고서에서는
    #
    # 당기 = 2023년 연간
    # 전기 = 2022년 연간

    annual_current_col = "당기"
    annual_previous_col = "전기"


    # --------------------------------------------------------
    # 연간 영업이익
    # --------------------------------------------------------

    annual_op = extract_item(
        annual,
        "operating",
        [
            annual_current_col,
            annual_previous_col
        ]
    )

    annual_op = annual_op.rename(
        columns={
            annual_current_col:
                "영업이익_2023연간",

            annual_previous_col:
                "영업이익_2022연간"
        }
    )


    # --------------------------------------------------------
    # 연간 매출
    # --------------------------------------------------------

    annual_rev = extract_item(
        annual,
        "revenue",
        [
            annual_current_col,
            annual_previous_col
        ]
    )

    annual_rev = annual_rev.rename(
        columns={
            annual_current_col:
                "매출_2023연간",

            annual_previous_col:
                "매출_2022연간"
        }
    )


    # ========================================================
    # 11-2. 2023 3분기보고서
    # ========================================================

    q3 = read_dart_file(
        q3_path
    )

    q3 = prepare_base(
        q3
    )


    # 공백 제거되어 있으므로
    q3_current_col = (
        "당기 3분기 누적"
    )

    q3_previous_col = (
        "전기 3분기 누적"
    )


    # --------------------------------------------------------
    # Q1~Q3 누적 영업이익
    # --------------------------------------------------------

    q3_op = extract_item(
        q3,
        "operating",
        [
            q3_current_col,
            q3_previous_col
        ]
    )

    q3_op = q3_op.rename(
        columns={
            q3_current_col:
                "영업이익_2023_9M",

            q3_previous_col:
                "영업이익_2022_9M"
        }
    )


    # --------------------------------------------------------
    # Q1~Q3 누적 매출
    # --------------------------------------------------------

    q3_rev = extract_item(
        q3,
        "revenue",
        [
            q3_current_col,
            q3_previous_col
        ]
    )

    q3_rev = q3_rev.rename(
        columns={
            q3_current_col:
                "매출_2023_9M",

            q3_previous_col:
                "매출_2022_9M"
        }
    )


    # ========================================================
    # 11-3. 기업별 결합
    # ========================================================

    meta = get_meta(
        annual
    )


    data = (
        meta

        .merge(
            annual_op,
            on="기업키",
            how="inner"
        )

        .merge(
            annual_rev,
            on="기업키",
            how="inner"
        )

        .merge(
            q3_op,
            on="기업키",
            how="inner"
        )

        .merge(
            q3_rev,
            on="기업키",
            how="inner"
        )
    )


    # ========================================================
    # 11-4. 숫자로 변환
    # ========================================================

    cols = [
        "영업이익_2023연간",
        "영업이익_2022연간",
        "매출_2023연간",
        "매출_2022연간",
        "영업이익_2023_9M",
        "영업이익_2022_9M",
        "매출_2023_9M",
        "매출_2022_9M"
    ]

    for col in cols:

        data[col] = (
            to_number(
                data[col]
            )
        )


    data = data.dropna(
        subset=cols
    )


    # ========================================================
    # ★ 11-5. Q4 단독값 계산
    # ========================================================

    # 2023Q4

    data["영업이익_당기"] = (
        data["영업이익_2023연간"]
        -
        data["영업이익_2023_9M"]
    )

    data["매출_당기"] = (
        data["매출_2023연간"]
        -
        data["매출_2023_9M"]
    )


    # 2022Q4

    data["영업이익_전년"] = (
        data["영업이익_2022연간"]
        -
        data["영업이익_2022_9M"]
    )

    data["매출_전년"] = (
        data["매출_2022연간"]
        -
        data["매출_2022_9M"]
    )


    return finalize_firm_data(
        data
    )


# ============================================================
# 12. 기업별 영업이익률 계산
# ============================================================

def finalize_firm_data(data):

    x = data.copy()


    # 숫자형 변환
    basic_cols = [
        "영업이익_당기",
        "영업이익_전년",
        "매출_당기",
        "매출_전년"
    ]

    for col in basic_cols:

        x[col] = (
            to_number(
                x[col]
            )
        )


    x = x.dropna(
        subset=basic_cols
    )


    # --------------------------------------------------------
    # 매출 0 이하 제외
    # --------------------------------------------------------

    x = x[
        (x["매출_당기"] > 0)
        &
        (x["매출_전년"] > 0)
    ].copy()


    # ========================================================
    # ★ 기업별 영업이익률
    # ========================================================

    x["전년영업이익률_%"] = (
        x["영업이익_전년"]
        /
        x["매출_전년"]
        * 100
    )


    x["당기영업이익률_%"] = (
        x["영업이익_당기"]
        /
        x["매출_당기"]
        * 100
    )


    x["변화_%p"] = (
        x["당기영업이익률_%"]
        -
        x["전년영업이익률_%"]
    )


    # ========================================================
    # 업종 대분류
    # ========================================================

    x["대분류"] = (
        x["업종"]
        .apply(
            broad_industry
        )
    )


    x = x.dropna(
        subset=["대분류"]
    )


    # 제조업 vs 비제조업

    x["제조업구분"] = np.where(
        x["대분류"]
        == "C.제조업",
        "제조업",
        "비제조업"
    )


    return x


# ============================================================
# 13. ★ IQR 이상치 제거
#
# 정상범위:
#
# Q1 - 1.5 × IQR
# ~
# Q3 + 1.5 × IQR
# ============================================================

def remove_outliers_iqr(data):

    x = data.copy()

    before = len(x)


    # ========================================================
    # 전년도
    # ========================================================

    prev_q1 = (
        x["전년영업이익률_%"]
        .quantile(0.25)
    )

    prev_median = (
        x["전년영업이익률_%"]
        .median()
    )

    prev_q3 = (
        x["전년영업이익률_%"]
        .quantile(0.75)
    )

    prev_iqr = (
        prev_q3 - prev_q1
    )

    prev_lower = (
        prev_q1
        - IQR_K * prev_iqr
    )

    prev_upper = (
        prev_q3
        + IQR_K * prev_iqr
    )


    # ========================================================
    # 당기
    # ========================================================

    curr_q1 = (
        x["당기영업이익률_%"]
        .quantile(0.25)
    )

    curr_median = (
        x["당기영업이익률_%"]
        .median()
    )

    curr_q3 = (
        x["당기영업이익률_%"]
        .quantile(0.75)
    )

    curr_iqr = (
        curr_q3 - curr_q1
    )

    curr_lower = (
        curr_q1
        - IQR_K * curr_iqr
    )

    curr_upper = (
        curr_q3
        + IQR_K * curr_iqr
    )


    # ========================================================
    # 둘 다 정상인 기업만
    # ========================================================

    normal_mask = (

        x["전년영업이익률_%"]
        .between(
            prev_lower,
            prev_upper,
            inclusive="both"
        )

        &

        x["당기영업이익률_%"]
        .between(
            curr_lower,
            curr_upper,
            inclusive="both"
        )
    )


    removed_data = (
        x[
            ~normal_mask
        ]
        .copy()
    )


    x = (
        x[
            normal_mask
        ]
        .copy()
    )


    after = len(x)

    removed = (
        before - after
    )

    removed_rate = (
        removed / before * 100
        if before > 0
        else 0
    )


    # 다시 계산

    x["변화_%p"] = (
        x["당기영업이익률_%"]
        -
        x["전년영업이익률_%"]
    )

    x["악화여부"] = (
        x["변화_%p"] < 0
    )


    info = {

        "원본기업수": before,

        "분석기업수": after,

        "제거기업수": removed,

        "제거율": removed_rate,


        "전년_Q1": prev_q1,

        "전년_중앙값": prev_median,

        "전년_Q3": prev_q3,

        "전년_IQR": prev_iqr,

        "전년하한": prev_lower,

        "전년상한": prev_upper,


        "당기_Q1": curr_q1,

        "당기_중앙값": curr_median,

        "당기_Q3": curr_q3,

        "당기_IQR": curr_iqr,

        "당기하한": curr_lower,

        "당기상한": curr_upper
    }


    return (
        x,
        removed_data,
        info
    )


# ============================================================
# 14. 그룹별 단순 평균
#
# ★ 모든 기업 가중치 = 1
# ============================================================

def summary_table(
    data,
    group_col
):

    firm = (
        data
        .drop_duplicates(
            subset=[
                "기업키",
                group_col
            ]
        )
        .copy()
    )


    result = (

        firm

        .groupby(
            group_col
        )

        .agg(

            기업수=(
                "기업키",
                "nunique"
            ),

            전년평균_=(
                "전년영업이익률_%",
                "mean"
            ),

            당기평균_=(
                "당기영업이익률_%",
                "mean"
            ),

            변화평균_=(
                "변화_%p",
                "mean"
            ),

            변화중앙값_=(
                "변화_%p",
                "median"
            ),

            악화기업비율_=(
                "악화여부",
                "mean"
            )
        )

        .reset_index()
    )


    result[
        "악화기업비율_"
    ] *= 100


    result = result.rename(

        columns={

            "전년평균_":
                "전년평균_%",

            "당기평균_":
                "당기평균_%",

            "변화평균_":
                "변화평균_%p",

            "변화중앙값_":
                "변화중앙값_%p",

            "악화기업비율_":
                "악화기업비율_%"
        }
    )


    cols = [

        "전년평균_%",

        "당기평균_%",

        "변화평균_%p",

        "변화중앙값_%p",

        "악화기업비율_%"
    ]


    result[cols] = (
        result[cols]
        .round(2)
    )


    return result


# ============================================================
# 15. 색상
# ============================================================

def change_color(v):

    if pd.isna(v):
        return ""

    if v < 0:

        return (
            "background-color:#fdeaea;"
            "color:#d32f2f;"
            "font-weight:bold;"
        )

    elif v > 0:

        return (
            "background-color:#e7f6ec;"
            "color:#087b35;"
            "font-weight:bold;"
        )

    return ""


def sample_color(v):

    if v == "소표본":

        return (
            "background-color:#fff3cd;"
            "color:#8a6500;"
            "font-weight:bold;"
        )

    return (
        "background-color:#e7f0ff;"
        "color:#2457a6;"
    )


# ============================================================
# 16. 예쁜 표
# ============================================================

def pretty_table(
    df,
    title,
    industry=False
):

    x = df.copy()


    if industry:

        x = (
            x
            .reset_index(drop=True)
        )

        x.insert(
            0,
            "순위",
            np.arange(
                1,
                len(x) + 1
            )
        )


    rename_map = {

        "제조업구분":
            "구분",

        "대분류":
            "업종",

        "기업수":
            "기업 수",

        "전년평균_%":
            "전년 영업이익률",

        "당기평균_%":
            "당기 영업이익률",

        "변화평균_%p":
            "평균 변화",

        "변화중앙값_%p":
            "중앙값 변화",

        "악화기업비율_%":
            "악화 기업 비율",

        "상태":
            "표본 상태"
    }


    x = x.rename(
        columns=rename_map
    )


    fmt = {}


    if "기업 수" in x.columns:
        fmt["기업 수"] = "{:,.0f}"

    if "전년 영업이익률" in x.columns:
        fmt["전년 영업이익률"] = "{:.2f}%"

    if "당기 영업이익률" in x.columns:
        fmt["당기 영업이익률"] = "{:.2f}%"

    if "평균 변화" in x.columns:
        fmt["평균 변화"] = "{:+.2f}%p"

    if "중앙값 변화" in x.columns:
        fmt["중앙값 변화"] = "{:+.2f}%p"

    if "악화 기업 비율" in x.columns:
        fmt["악화 기업 비율"] = "{:.1f}%"


    styler = (

        x.style

        .hide(axis="index")

        .format(
            fmt,
            na_rep="-"
        )

        .set_caption(
            title
        )

        .set_table_styles(
            [

                {
                    "selector": "caption",
                    "props": [
                        ("caption-side", "top"),
                        ("font-size", "20px"),
                        ("font-weight", "700"),
                        ("text-align", "left"),
                        ("padding", "14px 4px")
                    ]
                },

                {
                    "selector": "thead th",
                    "props": [
                        ("background-color", "#343a40"),
                        ("color", "white"),
                        ("font-weight", "700"),
                        ("text-align", "center"),
                        ("padding", "10px 12px")
                    ]
                },

                {
                    "selector": "tbody td",
                    "props": [
                        ("padding", "10px 12px"),
                        ("text-align", "center"),
                        (
                            "border-bottom",
                            "1px solid #aaaaaa"
                        )
                    ]
                },

                {
                    "selector": "table",
                    "props": [
                        (
                            "border-collapse",
                            "collapse"
                        ),
                        ("font-size", "15px"),
                        ("width", "100%")
                    ]
                }
            ]
        )
    )


    change_cols = [

        c for c in
        [
            "평균 변화",
            "중앙값 변화"
        ]

        if c in x.columns
    ]


    if change_cols:

        styler = styler.applymap(
            change_color,
            subset=change_cols
        )


    if "표본 상태" in x.columns:

        styler = styler.applymap(
            sample_color,
            subset=["표본 상태"]
        )


    if "업종" in x.columns:

        styler = styler.set_properties(
            subset=["업종"],
            **{
                "text-align": "left",
                "font-weight": "600"
            }
        )


    if "구분" in x.columns:

        styler = styler.set_properties(
            subset=["구분"],
            **{
                "font-weight": "700"
            }
        )


    return styler


# ============================================================
# 17. 결과 출력
# ============================================================

def analyze_data(
    raw,
    title
):

    # ========================================================
    # IQR 제거
    # ========================================================

    (
        data,
        removed_data,
        info

    ) = remove_outliers_iqr(
        raw
    )


    print("\n" + "=" * 80)

    print(title)

    print("=" * 80)


    print(
        "\n[IQR 이상치 제거]"
    )

    print(
        "기준 : "
        "Q1 - 1.5×IQR "
        "~ "
        "Q3 + 1.5×IQR"
    )


    print(
        f"\n원본 기업 수 : "
        f"{info['원본기업수']:,}"
    )

    print(
        f"분석 기업 수 : "
        f"{info['분석기업수']:,}"
    )

    print(
        f"제거 기업 수 : "
        f"{info['제거기업수']:,}"
    )

    print(
        f"제거율 : "
        f"{info['제거율']:.2f}%"
    )


    print("\n[전년 영업이익률]")

    print(
        f"Q1 = "
        f"{info['전년_Q1']:.2f}%"
    )

    print(
        f"중앙값 = "
        f"{info['전년_중앙값']:.2f}%"
    )

    print(
        f"Q3 = "
        f"{info['전년_Q3']:.2f}%"
    )

    print(
        f"IQR = "
        f"{info['전년_IQR']:.2f}%p"
    )

    print(
        f"정상범위 = "
        f"{info['전년하한']:.2f}%"
        " ~ "
        f"{info['전년상한']:.2f}%"
    )


    print("\n[당기 영업이익률]")

    print(
        f"Q1 = "
        f"{info['당기_Q1']:.2f}%"
    )

    print(
        f"중앙값 = "
        f"{info['당기_중앙값']:.2f}%"
    )

    print(
        f"Q3 = "
        f"{info['당기_Q3']:.2f}%"
    )

    print(
        f"IQR = "
        f"{info['당기_IQR']:.2f}%p"
    )

    print(
        f"정상범위 = "
        f"{info['당기하한']:.2f}%"
        " ~ "
        f"{info['당기상한']:.2f}%"
    )


    # ========================================================
    # 제조업 vs 비제조업
    # ========================================================

    manufacturing = summary_table(
        data,
        "제조업구분"
    )


    order = {
        "제조업": 0,
        "비제조업": 1
    }


    manufacturing["_order"] = (
        manufacturing[
            "제조업구분"
        ]
        .map(order)
    )


    manufacturing = (
        manufacturing
        .sort_values("_order")
        .drop(
            columns="_order"
        )
        .reset_index(
            drop=True
        )
    )


    # ========================================================
    # 업종별
    # ========================================================

    industry = summary_table(
        data,
        "대분류"
    )


    industry["상태"] = np.where(

        industry["기업수"]
        >= MIN_FIRMS,

        "집계 가능",

        "소표본"
    )


    # 영업이익률 악화 순

    industry = (
        industry
        .sort_values(
            [
                "변화평균_%p",
                "기업수"
            ],
            ascending=[
                True,
                False
            ]
        )
        .reset_index(
            drop=True
        )
    )


    # ========================================================
    # 화면 출력
    # ========================================================

    display(

        pretty_table(
            manufacturing,
            title=(
                f"📊 {title} | "
                "제조업 vs 비제조업"
            )
        )
    )


    display(

        pretty_table(
            industry,
            title=(
                f"📉 {title} | "
                "업종별 영업이익률 변화"
            ),
            industry=True
        )
    )


    return {

        "원본기업":
            raw,

        "분석기업":
            data,

        "제거기업":
            removed_data,

        "제조업비교":
            manufacturing,

        "업종별":
            industry,

        "IQR정보":
            info
    }


# ============================================================
# 18. ★ 홍해 2023Q4 vs 2022Q4
#
# 2023 3분기보고서가 있어야 실행 가능
# ============================================================

q4_raw = make_q4_data(

    ANNUAL_2023_FILE,

    Q3_2023_FILE
)


redsea_q4_result = analyze_data(

    q4_raw,

    title=
        "홍해 | 2023Q4 vs 2022Q4"
)


# ============================================================
# 19. ★ 홍해 2024Q1 vs 2023Q1
# ============================================================

q1_raw = make_q1_data(
    Q1_2024_FILE
)


redsea_q1_result = analyze_data(

    q1_raw,

    title=
        "홍해 | 2024Q1 vs 2023Q1"
)


홍해 | 2023Q4 vs 2022Q4

[IQR 이상치 제거]
기준 : Q1 - 1.5×IQR ~ Q3 + 1.5×IQR

원본 기업 수 : 1,776
분석 기업 수 : 1,398
제거 기업 수 : 378
제거율 : 21.28%

[전년 영업이익률]
Q1 = -6.01%
중앙값 = 2.65%
Q3 = 8.41%
IQR = 14.42%p
정상범위 = -27.63% ~ 30.03%

[당기 영업이익률]
Q1 = -7.38%
중앙값 = 1.96%
Q3 = 7.50%
IQR = 14.88%p
정상범위 = -29.69% ~ 29.81%


구분,기업 수,전년 영업이익률,당기 영업이익률,평균 변화,중앙값 변화,악화 기업 비율
제조업,919,3.32%,2.35%,-0.98%p,-0.56%p,53.6%
비제조업,479,4.21%,3.38%,-0.84%p,-0.19%p,52.8%


순위,업종,기업 수,전년 영업이익률,당기 영업이익률,평균 변화,중앙값 변화,악화 기업 비율,표본 상태
1,"E.수도, 하수 및 폐기물 처리, 원료 재생업",1,9.96%,6.70%,-3.25%p,-3.25%p,100.0%,소표본
2,G.도매 및 소매업,94,2.30%,0.41%,-1.89%p,-0.30%p,55.3%,집계 가능
3,J.정보통신업,178,6.26%,4.76%,-1.50%p,-0.73%p,55.1%,집계 가능
4,"R.예술, 스포츠 및 여가관련 서비스업",3,6.93%,5.65%,-1.28%p,-3.35%p,66.7%,소표본
5,C.제조업,919,3.32%,2.35%,-0.98%p,-0.56%p,53.6%,집계 가능
6,F.건설업,39,3.39%,2.64%,-0.75%p,-0.04%p,51.3%,집계 가능
7,H.운수 및 창고업,20,4.99%,4.52%,-0.48%p,-0.39%p,55.0%,집계 가능
8,P.교육 서비스업,7,4.80%,4.49%,-0.31%p,-2.12%p,71.4%,집계 가능
9,K.금융 및 보험업,57,4.39%,4.29%,-0.10%p,+0.48%p,47.4%,집계 가능
10,"M.전문, 과학 및 기술 서비스업",45,1.35%,1.47%,+0.12%p,-0.02%p,51.1%,집계 가능



홍해 | 2024Q1 vs 2023Q1

[IQR 이상치 제거]
기준 : Q1 - 1.5×IQR ~ Q3 + 1.5×IQR

원본 기업 수 : 1,868
분석 기업 수 : 1,524
제거 기업 수 : 344
제거율 : 18.42%

[전년 영업이익률]
Q1 = -7.02%
중앙값 = 2.57%
Q3 = 7.60%
IQR = 14.62%p
정상범위 = -28.95% ~ 29.53%

[당기 영업이익률]
Q1 = -7.43%
중앙값 = 2.74%
Q3 = 7.23%
IQR = 14.66%p
정상범위 = -29.42% ~ 29.22%


구분,기업 수,전년 영업이익률,당기 영업이익률,평균 변화,중앙값 변화,악화 기업 비율
제조업,"1,000",2.95%,3.15%,+0.21%p,+0.14%p,48.7%
비제조업,524,3.16%,2.23%,-0.93%p,-0.27%p,54.8%


순위,업종,기업 수,전년 영업이익률,당기 영업이익률,평균 변화,중앙값 변화,악화 기업 비율,표본 상태
1,"S.협회 및 단체, 수리 및 기타 개인 서비스업",2,14.54%,11.11%,-3.43%p,-3.43%p,50.0%,소표본
2,F.건설업,45,3.37%,1.56%,-1.81%p,-0.40%p,57.8%,집계 가능
3,I.숙박 및 음식점업,5,6.41%,4.99%,-1.42%p,-0.09%p,60.0%,집계 가능
4,J.정보통신업,185,1.21%,-0.17%,-1.38%p,-0.70%p,56.2%,집계 가능
5,"M.전문, 과학 및 기술 서비스업",48,2.64%,1.43%,-1.21%p,-0.36%p,54.2%,집계 가능
6,G.도매 및 소매업,108,2.75%,1.80%,-0.94%p,-0.35%p,56.5%,집계 가능
7,"R.예술, 스포츠 및 여가관련 서비스업",4,12.22%,11.28%,-0.94%p,-0.67%p,50.0%,소표본
8,P.교육 서비스업,7,5.45%,4.53%,-0.92%p,-1.84%p,71.4%,집계 가능
9,K.금융 및 보험업,61,6.21%,5.59%,-0.62%p,+0.20%p,45.9%,집계 가능
10,"E.수도, 하수 및 폐기물 처리, 원료 재생업",2,16.91%,16.38%,-0.53%p,-0.53%p,50.0%,소표본


In [27]:
# ============================================================
# 제조업 세부 업종 분석
# 기존 분석 결과를 그대로 활용
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display


# ============================================================
# 1. 제조업 세부 분류
# KSIC 2자리 기준 10~34
# ============================================================

MANUFACTURING_DETAIL = {

    10: "10.식료품 제조업",

    11: "11.음료 제조업",

    12: "12.담배 제조업",

    13: "13.섬유제품 제조업 (의복 제외)",

    14: "14.의복·의복 액세서리 및 모피제품 제조업",

    15: "15.가죽·가방 및 신발 제조업",

    16: "16.목재 및 나무제품 제조업 (가구 제외)",

    17: "17.펄프·종이 및 종이제품 제조업",

    18: "18.인쇄 및 기록매체 복제업",

    19: "19.코크스·연탄 및 석유정제품 제조업",

    20: "20.화학물질 및 화학제품 제조업 (의약품 제외)",

    21: "21.의료용 물질 및 의약품 제조업",

    22: "22.고무 및 플라스틱제품 제조업",

    23: "23.비금속 광물제품 제조업",

    24: "24.1차 금속 제조업",

    25: "25.금속가공제품 제조업 (기계·가구 제외)",

    26: "26.전자부품·컴퓨터·영상·음향 및 통신장비 제조업",

    27: "27.의료·정밀·광학기기 및 시계 제조업",

    28: "28.전기장비 제조업",

    29: "29.기타 기계 및 장비 제조업",

    30: "30.자동차 및 트레일러 제조업",

    31: "31.기타 운송장비 제조업",

    32: "32.가구 제조업",

    33: "33.기타 제품 제조업",

    34: "34.산업용 기계 및 장비 수리업"
}


# ============================================================
# 2. 기존 3자리 업종코드 → 제조업 2자리 코드
# ============================================================

def get_manufacturing_code(code):

    try:

        code = int(float(code))

    except:
        return np.nan


    # 예:
    # 262 -> 26
    # 272 -> 27
    # 292 -> 29
    # 301 -> 30

    if code >= 100:

        two_digit = code // 10

    else:

        two_digit = code


    # 제조업만
    if 10 <= two_digit <= 34:

        return two_digit


    return np.nan


# ============================================================
# 3. 제조업 세부업종별 분석
# ============================================================

def manufacturing_detail_summary(
    data,
    title,
    min_firms=5
):

    x = data.copy()


    # ========================================================
    # 제조업 10~34만 추출
    # ========================================================

    x["제조업코드"] = (
        x["업종"]
        .apply(get_manufacturing_code)
    )


    x = (
        x[
            x["제조업코드"].notna()
        ]
        .copy()
    )


    x["제조업코드"] = (
        x["제조업코드"]
        .astype(int)
    )


    # ========================================================
    # 세부 업종명
    # ========================================================

    x["제조업세부업종"] = (
        x["제조업코드"]
        .map(MANUFACTURING_DETAIL)
    )


    x = x.dropna(
        subset=["제조업세부업종"]
    )


    # ========================================================
    # 기업당 정확히 1행만
    # ========================================================

    firm = (

        x

        .drop_duplicates(
            subset=[
                "기업키",
                "제조업세부업종"
            ]
        )

        .copy()
    )


    # ========================================================
    # 세부 제조업별 단순 평균
    #
    # ★ 기업마다 가중치 = 1
    #
    # 기업A 4%
    # 기업B 5%
    # 기업C 6%
    # → 평균 5%
    # ========================================================

    result = (

        firm

        .groupby(
            "제조업세부업종"
        )

        .agg(

            기업수=(
                "기업키",
                "nunique"
            ),

            전년평균_=(
                "전년영업이익률_%",
                "mean"
            ),

            당기평균_=(
                "당기영업이익률_%",
                "mean"
            ),

            변화평균_=(
                "변화_%p",
                "mean"
            ),

            변화중앙값_=(
                "변화_%p",
                "median"
            ),

            악화기업비율_=(
                "악화여부",
                "mean"
            )

        )

        .reset_index()
    )


    # ========================================================
    # 악화 기업 비율 %
    # ========================================================

    result["악화기업비율_"] *= 100


    # ========================================================
    # 이름 정리
    # ========================================================

    result = result.rename(

        columns={

            "전년평균_":
                "전년평균_%",

            "당기평균_":
                "당기평균_%",

            "변화평균_":
                "변화평균_%p",

            "변화중앙값_":
                "변화중앙값_%p",

            "악화기업비율_":
                "악화기업비율_%"
        }
    )


    # ========================================================
    # 표본 수 표시
    # ========================================================

    result["상태"] = np.where(

        result["기업수"]
        >= min_firms,

        "집계 가능",

        "소표본"
    )


    # ========================================================
    # 영업이익률 가장 많이 낮아진 업종부터
    # ========================================================

    result = (

        result

        .sort_values(

            [
                "변화평균_%p",
                "기업수"
            ],

            ascending=[
                True,
                False
            ]
        )

        .reset_index(
            drop=True
        )
    )


    # ========================================================
    # 숫자 반올림
    # ========================================================

    numeric_cols = [

        "전년평균_%",

        "당기평균_%",

        "변화평균_%p",

        "변화중앙값_%p",

        "악화기업비율_%"
    ]


    result[numeric_cols] = (

        result[numeric_cols]

        .round(2)
    )


    # ========================================================
    # 예쁜 표 출력
    #
    # 앞에서 만든 pretty_table() 재사용
    # ========================================================

    display(

        pretty_manufacturing_table(

            result,

            title=(
                f"🏭 {title} | "
                "제조업 세부 업종별 영업이익률 변화"
            )
        )
    )


    return result


# ============================================================
# 4. 제조업 전용 예쁜 표
# ============================================================

def pretty_manufacturing_table(
    df,
    title
):

    x = df.copy()


    # 순위

    x.insert(

        0,

        "순위",

        np.arange(
            1,
            len(x) + 1
        )
    )


    # 컬럼명

    x = x.rename(

        columns={

            "제조업세부업종":
                "제조업 세부업종",

            "기업수":
                "기업 수",

            "전년평균_%":
                "전년 영업이익률",

            "당기평균_%":
                "당기 영업이익률",

            "변화평균_%p":
                "평균 변화",

            "변화중앙값_%p":
                "중앙값 변화",

            "악화기업비율_%":
                "악화 기업 비율",

            "상태":
                "표본 상태"
        }
    )


    # ========================================================
    # 표시 형식
    # ========================================================

    fmt = {

        "기업 수":
            "{:,.0f}",

        "전년 영업이익률":
            "{:.2f}%",

        "당기 영업이익률":
            "{:.2f}%",

        "평균 변화":
            "{:+.2f}%p",

        "중앙값 변화":
            "{:+.2f}%p",

        "악화 기업 비율":
            "{:.1f}%"
    }


    # ========================================================
    # 스타일
    # ========================================================

    styler = (

        x.style

        .hide(axis="index")

        .format(
            fmt,
            na_rep="-"
        )

        .set_caption(
            title
        )

        .set_table_styles(
            [

                {
                    "selector": "caption",

                    "props": [

                        ("caption-side", "top"),

                        ("font-size", "20px"),

                        ("font-weight", "700"),

                        ("text-align", "left"),

                        ("padding", "14px 4px")
                    ]
                },

                {
                    "selector": "thead th",

                    "props": [

                        (
                            "background-color",
                            "#343a40"
                        ),

                        (
                            "color",
                            "white"
                        ),

                        (
                            "font-weight",
                            "700"
                        ),

                        (
                            "text-align",
                            "center"
                        ),

                        (
                            "padding",
                            "10px 12px"
                        )
                    ]
                },

                {
                    "selector": "tbody td",

                    "props": [

                        (
                            "padding",
                            "9px 12px"
                        ),

                        (
                            "text-align",
                            "center"
                        ),

                        (
                            "border-bottom",
                            "1px solid #aaaaaa"
                        )
                    ]
                },

                {
                    "selector": "table",

                    "props": [

                        (
                            "border-collapse",
                            "collapse"
                        ),

                        (
                            "font-size",
                            "14px"
                        ),

                        (
                            "width",
                            "100%"
                        )
                    ]
                }
            ]
        )
    )


    # ========================================================
    # 변화 색상
    # ========================================================

    styler = styler.applymap(

        change_color,

        subset=[
            "평균 변화",
            "중앙값 변화"
        ]
    )


    # ========================================================
    # 표본 상태
    # ========================================================

    styler = styler.applymap(

        sample_color,

        subset=[
            "표본 상태"
        ]
    )


    # 업종명 왼쪽 정렬

    styler = styler.set_properties(

        subset=[
            "제조업 세부업종"
        ],

        **{
            "text-align": "left",
            "font-weight": "600"
        }
    )


    return styler

In [28]:
# ============================================================
# 5. 홍해 2023Q4 vs 2022Q4
# ============================================================

redsea_q4_manufacturing = (
    manufacturing_detail_summary(

        redsea_q4_result[
            "분석기업"
        ],

        "홍해 | 2023Q4 vs 2022Q4"
    )
)


# ============================================================
# 6. 홍해 2024Q1 vs 2023Q1
# ============================================================

redsea_q1_manufacturing = (
    manufacturing_detail_summary(

        redsea_q1_result[
            "분석기업"
        ],

        "홍해 | 2024Q1 vs 2023Q1"
    )
)


# ============================================================
# 7. 호르무즈 2026Q1 vs 2025Q1
# ============================================================

hormuz_q1_manufacturing = (
    manufacturing_detail_summary(

        q1_result[
            "분석기업"
        ],

        "호르무즈 | 2026Q1 vs 2025Q1"
    )
)


# ============================================================
# 8. 호르무즈 2026Q2 vs 2025Q2
# ============================================================

hormuz_q2_manufacturing = (
    manufacturing_detail_summary(

        q2_result[
            "분석기업"
        ],

        "호르무즈 | 2026Q2 vs 2025Q2"
    )
)

순위,제조업 세부업종,기업 수,전년 영업이익률,당기 영업이익률,평균 변화,중앙값 변화,악화 기업 비율,표본 상태
1,33.기타 제품 제조업,7,3.08%,-3.16%,-6.24%p,-8.28%p,57.1%,집계 가능
2,14.의복·의복 액세서리 및 모피제품 제조업,23,6.12%,2.66%,-3.46%p,-3.37%p,60.9%,집계 가능
3,27.의료·정밀·광학기기 및 시계 제조업,41,8.75%,5.73%,-3.02%p,-3.76%p,58.5%,집계 가능
4,29.기타 기계 및 장비 제조업,124,6.01%,3.64%,-2.38%p,-2.24%p,59.7%,집계 가능
5,32.가구 제조업,5,-2.89%,-4.85%,-1.97%p,-0.71%p,80.0%,집계 가능
6,30.자동차 및 트레일러 제조업,77,3.25%,1.54%,-1.71%p,-1.35%p,66.2%,집계 가능
7,24.1차 금속 제조업,50,1.90%,0.29%,-1.61%p,-1.06%p,62.0%,집계 가능
8,25.금속가공제품 제조업 (기계·가구 제외),21,4.55%,3.05%,-1.50%p,+0.75%p,42.9%,집계 가능
9,15.가죽·가방 및 신발 제조업,4,7.56%,6.38%,-1.18%p,+0.69%p,25.0%,소표본
10,21.의료용 물질 및 의약품 제조업,63,3.34%,2.16%,-1.18%p,-0.38%p,55.6%,집계 가능


순위,제조업 세부업종,기업 수,전년 영업이익률,당기 영업이익률,평균 변화,중앙값 변화,악화 기업 비율,표본 상태
1,12.담배 제조업,1,22.68%,18.31%,-4.37%p,-4.37%p,100.0%,소표본
2,24.1차 금속 제조업,57,4.18%,1.45%,-2.73%p,-1.93%p,68.4%,집계 가능
3,25.금속가공제품 제조업 (기계·가구 제외),25,6.11%,4.30%,-1.81%p,-1.74%p,60.0%,집계 가능
4,15.가죽·가방 및 신발 제조업,5,7.70%,6.64%,-1.06%p,+1.85%p,40.0%,집계 가능
5,14.의복·의복 액세서리 및 모피제품 제조업,23,3.65%,2.63%,-1.01%p,-0.15%p,56.5%,집계 가능
6,30.자동차 및 트레일러 제조업,83,3.98%,3.21%,-0.77%p,-0.32%p,55.4%,집계 가능
7,29.기타 기계 및 장비 제조업,126,3.02%,2.50%,-0.52%p,-0.99%p,54.0%,집계 가능
8,23.비금속 광물제품 제조업,22,5.00%,4.56%,-0.44%p,-0.31%p,54.5%,집계 가능
9,31.기타 운송장비 제조업,19,2.45%,2.22%,-0.23%p,+1.31%p,42.1%,집계 가능
10,20.화학물질 및 화학제품 제조업 (의약품 제외),113,4.84%,5.05%,+0.21%p,+0.52%p,46.9%,집계 가능


순위,제조업 세부업종,기업 수,전년 영업이익률,당기 영업이익률,평균 변화,중앙값 변화,악화 기업 비율,표본 상태
1,13.섬유제품 제조업 (의복 제외),9,5.16%,-1.52%,-6.68%p,+0.07%p,44.4%,집계 가능
2,32.가구 제조업,5,2.16%,-2.07%,-4.24%p,+1.26%p,40.0%,집계 가능
3,18.인쇄 및 기록매체 복제업,1,9.51%,7.25%,-2.25%p,-2.25%p,100.0%,소표본
4,15.가죽·가방 및 신발 제조업,6,6.57%,4.89%,-1.68%p,-1.95%p,66.7%,집계 가능
5,21.의료용 물질 및 의약품 제조업,78,4.38%,3.01%,-1.37%p,-0.53%p,52.6%,집계 가능
6,33.기타 제품 제조업,8,-0.07%,-1.42%,-1.35%p,+1.51%p,37.5%,집계 가능
7,10.식료품 제조업,58,4.11%,2.97%,-1.14%p,-0.19%p,56.9%,집계 가능
8,30.자동차 및 트레일러 제조업,92,3.82%,2.77%,-1.05%p,-0.39%p,56.5%,집계 가능
9,11.음료 제조업,11,8.03%,7.01%,-1.02%p,-0.91%p,72.7%,집계 가능
10,17.펄프·종이 및 종이제품 제조업,18,-0.35%,-1.02%,-0.67%p,-1.42%p,72.2%,집계 가능


순위,제조업 세부업종,기업 수,전년 영업이익률,당기 영업이익률,평균 변화,중앙값 변화,악화 기업 비율,표본 상태
1,11.음료 제조업,11,10.32%,7.14%,-3.18%p,-2.20%p,72.7%,집계 가능
2,32.가구 제조업,5,-2.60%,-4.19%,-1.59%p,+0.25%p,40.0%,집계 가능
3,28.전기장비 제조업,59,3.49%,2.53%,-0.96%p,-0.17%p,52.5%,집계 가능
4,22.고무 및 플라스틱제품 제조업,43,4.47%,4.38%,-0.09%p,+0.10%p,48.8%,집계 가능
5,30.자동차 및 트레일러 제조업,89,4.05%,3.98%,-0.07%p,-0.30%p,52.8%,집계 가능
6,10.식료품 제조업,59,4.85%,4.96%,+0.12%p,+0.10%p,49.1%,집계 가능
7,21.의료용 물질 및 의약품 제조업,75,4.76%,4.98%,+0.21%p,+0.91%p,46.7%,집계 가능
8,23.비금속 광물제품 제조업,21,4.61%,5.54%,+0.93%p,+0.70%p,42.9%,집계 가능
9,25.금속가공제품 제조업 (기계·가구 제외),25,3.27%,4.29%,+1.02%p,+0.88%p,44.0%,집계 가능
10,33.기타 제품 제조업,6,5.40%,6.43%,+1.03%p,+2.22%p,16.7%,집계 가능


In [29]:
from pathlib import Path
import pandas as pd
import re

# ============================================================
# 1. 경로 설정
# ============================================================

# TXT 파일들이 들어있는 폴더
DATA_DIR = Path("datas/손익계산서(PL)")

# 저장 파일
OUT_PKL = DATA_DIR / "포괄손익계산서_2022_2026반기_통합.pkl"
OUT_CSV = DATA_DIR / "포괄손익계산서_2022_2026반기_통합.csv"


# ============================================================
# 2. 파일명 뒤 다운로드 날짜 제거
# ============================================================

def clean_filename(filename):
    """
    예:
    2025_1분기보고서_03_포괄손익계산서_연결_20260814(1).txt
        ↓
    2025_1분기보고서_03_포괄손익계산서_연결

    2024_반기보고서_03_포괄손익계산서_20260909.txt
        ↓
    2024_반기보고서_03_포괄손익계산서
    """

    stem = Path(filename).stem

    # 맨 뒤의 _YYYYMMDD 제거
    # 뒤에 (1), (2) 등이 붙어 있어도 제거
    stem = re.sub(
        r'_\d{8}(?:\(\d+\))?$',
        '',
        stem
    )

    return stem


# ============================================================
# 3. 원하는 기간인지 확인
# ============================================================

def is_target_file(path):
    name = clean_filename(path.name)

    # --------------------------------------------------------
    # 포괄손익계산서 파일만
    # --------------------------------------------------------
    if "_03_포괄손익계산서" not in name:
        return False

    # 혹시 다른 비슷한 파일이 있다면 방지
    pattern = (
        r"^(20\d{2})_"
        r"(사업보고서|1분기보고서|반기보고서|3분기보고서)_"
        r"03_포괄손익계산서"
        r"(_연결)?$"
    )

    m = re.match(pattern, name)

    if m is None:
        return False

    year = int(m.group(1))
    report = m.group(2)

    # --------------------------------------------------------
    # 2022 ~ 2025 : 모든 보고서 허용
    # --------------------------------------------------------
    if 2022 <= year <= 2025:
        return True

    # --------------------------------------------------------
    # 2026 : 반기까지
    # 사업보고서 / 1분기 / 반기만 포함
    # --------------------------------------------------------
    if year == 2026:
        return report in [
            "사업보고서",
            "1분기보고서",
            "반기보고서"
        ]

    return False


# ============================================================
# 4. 파일 정보 추출
# ============================================================

def get_file_info(path):

    clean_name = clean_filename(path.name)

    pattern = (
        r"^(20\d{2})_"
        r"(사업보고서|1분기보고서|반기보고서|3분기보고서)_"
        r"03_포괄손익계산서"
        r"(_연결)?$"
    )

    m = re.match(pattern, clean_name)

    year = int(m.group(1))
    report = m.group(2)

    if m.group(3) == "_연결":
        statement_type = "연결"
    else:
        statement_type = "별도"

    # 분석하기 편하도록 분기 번호
    quarter_map = {
        "사업보고서": 4,
        "1분기보고서": 1,
        "반기보고서": 2,
        "3분기보고서": 3,
    }

    quarter = quarter_map[report]

    return {
        "연도": year,
        "분기": quarter,
        "파일보고서종류": report,
        "재무구분": statement_type,
        "원본파일명": path.name,
        "정리파일명": clean_name,
    }


# ============================================================
# 5. 대상 파일 찾기
# ============================================================

all_txt = list(DATA_DIR.glob("*.txt"))

target_files = [
    f for f in all_txt
    if is_target_file(f)
]

print("=" * 70)
print(f"찾은 파일 수: {len(target_files):,}개")
print("=" * 70)

for f in sorted(target_files):
    print(clean_filename(f.name))


# ============================================================
# 6. 같은 파일을 여러 번 다운로드한 경우 중복 제거
# ============================================================

# 예:
# 2025_1분기보고서_03_포괄손익계산서_20260814.txt
# 2025_1분기보고서_03_포괄손익계산서_20260814(1).txt
#
# 이런 경우 둘 다 합치면 데이터가 2배가 되므로
# 정리된 파일명 기준 1개만 사용

unique_files = {}

for f in target_files:

    key = clean_filename(f.name)

    # 같은 key가 여러 개면 수정시간이 가장 최근 파일 사용
    if key not in unique_files:
        unique_files[key] = f

    else:
        old = unique_files[key]

        if f.stat().st_mtime > old.stat().st_mtime:
            unique_files[key] = f


target_files = list(unique_files.values())

print()
print("=" * 70)
print(f"중복 제거 후 파일 수: {len(target_files):,}개")
print("=" * 70)


# ============================================================
# 7. 파일 읽어서 합치기
# ============================================================

dfs = []

for i, file in enumerate(sorted(target_files), start=1):

    info = get_file_info(file)

    print(
        f"[{i}/{len(target_files)}] "
        f"{info['연도']} "
        f"{info['파일보고서종류']} "
        f"{info['재무구분']} 읽는 중..."
    )

    try:
        df = pd.read_csv(
            file,
            sep="\t",
            encoding="cp949",
            dtype=str,
            low_memory=False
        )

    except UnicodeDecodeError:

        # 일부 파일에 특수문자가 들어간 경우 대비
        df = pd.read_csv(
            file,
            sep="\t",
            encoding="cp949",
            encoding_errors="replace",
            dtype=str,
            low_memory=False
        )

    # --------------------------------------------------------
    # 컬럼명 앞뒤 공백 제거
    # --------------------------------------------------------
    df.columns = [
        str(col).strip()
        for col in df.columns
    ]

    # --------------------------------------------------------
    # 완전히 빈 컬럼 삭제
    #
    # 사업보고서 파일에는
    # Unnamed: 12, Unnamed: 14 같은 빈 열이 존재함
    # --------------------------------------------------------
    empty_cols = [
        col for col in df.columns
        if col.startswith("Unnamed:")
        and df[col].isna().all()
    ]

    df = df.drop(columns=empty_cols)

    # --------------------------------------------------------
    # 파일 출처 정보 추가
    # --------------------------------------------------------
    df.insert(0, "연도", info["연도"])
    df.insert(1, "분기", info["분기"])
    df.insert(2, "파일보고서종류", info["파일보고서종류"])
    df.insert(3, "재무구분", info["재무구분"])
    df.insert(4, "원본파일명", info["원본파일명"])

    dfs.append(df)

    print(f"    → {len(df):,}행")


# ============================================================
# 8. 전체 합치기
# ============================================================

if len(dfs) == 0:
    raise ValueError(
        "조건에 맞는 파일을 찾지 못했습니다. "
        "DATA_DIR 경로와 파일명을 확인하세요."
    )

combined = pd.concat(
    dfs,
    ignore_index=True,
    sort=False
)


# ============================================================
# 9. 정렬
# ============================================================

combined = combined.sort_values(
    ["연도", "분기", "재무구분", "종목코드", "항목코드"],
    na_position="last"
).reset_index(drop=True)


# ============================================================
# 10. 저장
# ============================================================

# 대용량 분석에는 PKL 권장
combined.to_pickle(OUT_PKL)

print()
print(f"PKL 저장 완료")
print(OUT_PKL)


# CSV도 필요하면 저장
# 행이 아주 많으면 시간이 꽤 걸릴 수 있음
combined.to_csv(
    OUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

print()
print(f"CSV 저장 완료")
print(OUT_CSV)


# ============================================================
# 11. 결과 확인
# ============================================================

print()
print("=" * 70)
print("통합 완료")
print("=" * 70)

print(f"전체 행 수 : {len(combined):,}")
print(f"전체 열 수 : {len(combined.columns):,}")

print("\n연도/보고서/재무구분별 행 수")

summary = (
    combined
    .groupby(
        ["연도", "파일보고서종류", "재무구분"],
        dropna=False
    )
    .size()
    .reset_index(name="행수")
)

print(summary.to_string(index=False))

print("\n컬럼 목록")
print(combined.columns.tolist())

찾은 파일 수: 36개
2022_1분기보고서_03_포괄손익계산서
2022_1분기보고서_03_포괄손익계산서_연결
2022_3분기보고서_03_포괄손익계산서
2022_3분기보고서_03_포괄손익계산서_연결
2022_반기보고서_03_포괄손익계산서
2022_반기보고서_03_포괄손익계산서_연결
2022_사업보고서_03_포괄손익계산서
2022_사업보고서_03_포괄손익계산서_연결
2023_1분기보고서_03_포괄손익계산서
2023_1분기보고서_03_포괄손익계산서_연결
2023_3분기보고서_03_포괄손익계산서
2023_3분기보고서_03_포괄손익계산서_연결
2023_반기보고서_03_포괄손익계산서
2023_반기보고서_03_포괄손익계산서_연결
2023_사업보고서_03_포괄손익계산서
2023_사업보고서_03_포괄손익계산서_연결
2024_1분기보고서_03_포괄손익계산서
2024_1분기보고서_03_포괄손익계산서_연결
2024_3분기보고서_03_포괄손익계산서
2024_3분기보고서_03_포괄손익계산서_연결
2024_반기보고서_03_포괄손익계산서
2024_반기보고서_03_포괄손익계산서_연결
2024_사업보고서_03_포괄손익계산서
2024_사업보고서_03_포괄손익계산서_연결
2025_1분기보고서_03_포괄손익계산서
2025_1분기보고서_03_포괄손익계산서_연결
2025_3분기보고서_03_포괄손익계산서
2025_3분기보고서_03_포괄손익계산서_연결
2025_반기보고서_03_포괄손익계산서
2025_반기보고서_03_포괄손익계산서_연결
2025_사업보고서_03_포괄손익계산서
2025_사업보고서_03_포괄손익계산서_연결
2026_1분기보고서_03_포괄손익계산서
2026_1분기보고서_03_포괄손익계산서_연결
2026_반기보고서_03_포괄손익계산서
2026_반기보고서_03_포괄손익계산서_연결

중복 제거 후 파일 수: 36개
[1/36] 2022 1분기보고서 별도 읽는 중...
    → 43,925행
[2/36] 2022 1분기보고서 연결 읽는 중...
    → 50,162행
[3/36] 2022 3분기보

In [31]:
df1 = combined.head(1000)

df1.to_csv(
    OUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

In [35]:
from pathlib import Path
import pandas as pd
import re
import unicodedata


# ============================================================
# 0. 경로 설정
# ============================================================

DATA_DIR = Path("datas")

# 재무 원본
FIN_PATH = DATA_DIR / "포괄손익계산서_2022_2026반기_통합.pkl"

# 결과
OUT_PKL = DATA_DIR / "포괄손익계산서_2022_2026반기_기업규모분류.pkl"
OUT_COMPANY = DATA_DIR / "연도별_기업규모분류표.csv"


# ============================================================
# 1. 회사명 정규화
# ============================================================

def normalize_company_name(x):
    """
    회사명 비교용 정규화

    예)
    삼성전자(주)
    (주)삼성전자
    삼성전자 주식회사
        ↓
    삼성전자
    """

    if pd.isna(x):
        return ""

    x = str(x).strip()

    # 유니코드 통일
    x = unicodedata.normalize("NFKC", x)

    # 영문 통일
    x = x.upper()

    # 법인 표현 제거
    remove_words = [
        "주식회사",
        "(주)",
        "㈜",
        "유한회사",
        "(유)",
        "㈲",
        "합자회사",
        "합명회사",
        "유한책임회사",

        "CO., LTD.",
        "CO.,LTD.",
        "CO.LTD.",
        "CO LTD",
        "CORPORATION",
        "CORP.",
        "CORP",
        "LIMITED",
        "LTD.",
        "LTD",
        "INC.",
        "INC",
    ]

    for word in remove_words:
        x = x.replace(word.upper(), "")

    # 공백/일부 특수문자 제거
    x = re.sub(r"[\s\-\._,&·ㆍ]", "", x)

    # 괄호 제거
    x = re.sub(r"[\(\)\[\]\{\}]", "", x)

    return x.strip()


# ============================================================
# 2. 연도별 리스트 파일 찾기
# ============================================================

YEARS = range(2022, 2027)


def find_year_file(prefix, year):
    """
    아래처럼 뒤에 (1), (2)가 있어도 자동 검색

    대기업리스트(2022).xlsx
    대기업리스트(2022)(1).xlsx

    중견기업리스트(2022).xlsx
    중견기업리스트(2022)(1).xlsx
    """

    candidates = list(
        DATA_DIR.glob(f"{prefix}({year})*.xlsx")
    )

    if len(candidates) == 0:
        raise FileNotFoundError(
            f"{year}년 {prefix} 파일을 찾지 못했습니다.\n"
            f"예상 파일명: {prefix}({year}).xlsx"
        )

    # 여러 개 있으면 수정시간이 가장 최신인 파일 사용
    candidates = sorted(
        candidates,
        key=lambda x: x.stat().st_mtime,
        reverse=True
    )

    return candidates[0]


# ============================================================
# 3. 연도별 대기업 리스트 읽기
# ============================================================

def load_large_company_list(path):

    df = pd.read_excel(
        path,
        sheet_name="소속회사 개요",
        dtype=str
    )

    if "소속회사명" not in df.columns:
        raise ValueError(
            f"{path.name}: '소속회사명' 열이 없습니다."
        )

    result = df[["소속회사명"]].copy()

    result["회사명_정규화"] = (
        result["소속회사명"]
        .apply(normalize_company_name)
    )

    result = result[
        result["회사명_정규화"] != ""
    ]

    # 동일 회사가 여러 기업집단에 중복되어 있어도 한 번만 사용
    result = result.drop_duplicates(
        subset=["회사명_정규화"]
    )

    return result


# ============================================================
# 4. 연도별 중견기업 리스트 읽기
# ============================================================

def load_mid_company_list(path):

    # 중견기업 파일은 첫 시트 사용
    df = pd.read_excel(
        path,
        sheet_name=0,
        dtype=str
    )

    if "기업명" not in df.columns:
        raise ValueError(
            f"{path.name}: '기업명' 열이 없습니다."
        )

    result = df[["기업명"]].copy()

    result["회사명_정규화"] = (
        result["기업명"]
        .apply(normalize_company_name)
    )

    result = result[
        result["회사명_정규화"] != ""
    ]

    result = result.drop_duplicates(
        subset=["회사명_정규화"]
    )

    return result


# ============================================================
# 5. 2022~2026 연도별 기업규모 리스트 구축
# ============================================================

large_by_year = {}
mid_by_year = {}


print("=" * 70)
print("연도별 기업규모 리스트 불러오기")
print("=" * 70)


for year in YEARS:

    large_path = find_year_file(
        "대기업리스트",
        year
    )

    mid_path = find_year_file(
        "중견기업리스트",
        year
    )

    large = load_large_company_list(
        large_path
    )

    mid = load_mid_company_list(
        mid_path
    )

    # set으로 만들어 빠르게 조회
    large_by_year[year] = set(
        large["회사명_정규화"]
    )

    mid_by_year[year] = set(
        mid["회사명_정규화"]
    )

    # 둘 다 포함된 기업 확인
    overlap = (
        large_by_year[year]
        & mid_by_year[year]
    )

    print()
    print(f"[{year}]")
    print(f"대기업 : {len(large_by_year[year]):,}개")
    print(f"중견기업 : {len(mid_by_year[year]):,}개")
    print(f"중복 : {len(overlap):,}개")

    print(f"대기업 파일 : {large_path.name}")
    print(f"중견기업 파일 : {mid_path.name}")

    if len(overlap) > 0:
        print(
            "※ 대기업/중견기업 리스트에 동시에 있으면 "
            "'대기업'을 우선 적용"
        )


# ============================================================
# 6. 포괄손익계산서 불러오기
# ============================================================

print()
print("=" * 70)
print("포괄손익계산서 불러오는 중")
print("=" * 70)

df = pd.read_pickle(
    FIN_PATH
)

# 연도를 숫자로
df["연도"] = pd.to_numeric(
    df["연도"],
    errors="coerce"
).astype("Int64")


print(f"전체 행 수 : {len(df):,}")
print(f"회사 수 : {df['회사명'].nunique():,}")

print(
    "대상연도 :",
    sorted(
        df["연도"]
        .dropna()
        .unique()
        .tolist()
    )
)


# ============================================================
# 7. 회사명 정규화
# ============================================================

df["회사명_정규화"] = (
    df["회사명"]
    .apply(normalize_company_name)
)


# ============================================================
# 8. 연도 × 기업 고유 테이블 만들기
#
# 원본 190만 행에 매번 판별하지 않고
# 기업별로 한 번만 판별하기 때문에 훨씬 빠름
# ============================================================

company_year = (
    df[
        [
            "연도",
            "종목코드",
            "회사명",
            "회사명_정규화"
        ]
    ]
    .drop_duplicates(
        subset=[
            "연도",
            "종목코드",
            "회사명_정규화"
        ]
    )
    .reset_index(drop=True)
)


print()
print(
    f"연도×기업 판별 대상 : "
    f"{len(company_year):,}개"
)


# ============================================================
# 9. 연도별 기업규모 분류
# ============================================================

def classify_company(year, name):

    if pd.isna(year) or name == "":
        return "분류불가"

    year = int(year)

    # 해당 연도 리스트가 없으면 분류불가
    if year not in large_by_year:
        return "분류불가"

    # ------------------------------------------
    # 1순위 : 대기업
    # ------------------------------------------
    if name in large_by_year[year]:
        return "대기업"

    # ------------------------------------------
    # 2순위 : 중견기업
    # ------------------------------------------
    if name in mid_by_year[year]:
        return "중견기업"

    # ------------------------------------------
    # 둘 다 아니면 중소기업
    # ------------------------------------------
    return "중소기업"


company_year["기업규모"] = [
    classify_company(year, name)
    for year, name
    in zip(
        company_year["연도"],
        company_year["회사명_정규화"]
    )
]


# ============================================================
# 10. 분류 근거도 같이 저장
# ============================================================

def classification_source(year, name):

    if pd.isna(year) or name == "":
        return "분류불가"

    year = int(year)

    if year not in large_by_year:
        return "해당연도 리스트 없음"

    if name in large_by_year[year]:
        return f"{year} 대기업리스트"

    if name in mid_by_year[year]:
        return f"{year} 중견기업리스트"

    return f"{year} 대기업·중견기업 리스트 미포함"


company_year["분류근거"] = [
    classification_source(year, name)
    for year, name
    in zip(
        company_year["연도"],
        company_year["회사명_정규화"]
    )
]


# ============================================================
# 11. 분류 결과 확인
# ============================================================

summary = (
    company_year
    .groupby(
        ["연도", "기업규모"],
        dropna=False
    )
    .size()
    .reset_index(
        name="기업수"
    )
)


print()
print("=" * 70)
print("연도별 기업규모 분류 결과")
print("=" * 70)

print(
    summary.to_string(
        index=False
    )
)


# ============================================================
# 12. 원본 재무데이터에 기업규모 붙이기
# ============================================================

# 기존에 기업규모가 있다면 제거
for col in ["기업규모", "분류근거"]:

    if col in df.columns:
        df = df.drop(
            columns=col
        )


# 연도 + 종목코드 + 정규화 회사명으로 merge

df = df.merge(
    company_year[
        [
            "연도",
            "종목코드",
            "회사명_정규화",
            "기업규모",
            "분류근거"
        ]
    ],
    on=[
        "연도",
        "종목코드",
        "회사명_정규화"
    ],
    how="left",
    validate="many_to_one"
)


# ============================================================
# 13. 최종 결과 확인
# ============================================================

print()
print("=" * 70)
print("최종 재무데이터 기업규모 분포")
print("=" * 70)

print(
    df[
        [
            "연도",
            "종목코드",
            "회사명",
            "기업규모"
        ]
    ]
    .drop_duplicates()
    .groupby(
        ["연도", "기업규모"]
    )
    .size()
    .reset_index(name="기업수")
    .to_string(index=False)
)


# ============================================================
# 14. 같은 기업이 연도별로 어떻게 바뀌었는지 확인
# ============================================================

size_change = (
    company_year
    .groupby(
        ["종목코드", "회사명"]
    )["기업규모"]
    .nunique()
)

changed_codes = (
    size_change[
        size_change > 1
    ]
    .index
)


changed = (
    company_year
    .set_index(
        ["종목코드", "회사명"]
    )
    .loc[
        company_year
        .set_index(
            ["종목코드", "회사명"]
        )
        .index
        .isin(changed_codes)
    ]
    .reset_index()
    .sort_values(
        [
            "회사명",
            "연도"
        ]
    )
)


print()
print("=" * 70)
print("연도별 기업규모가 달라진 기업 예시")
print("=" * 70)

print(
    changed[
        [
            "연도",
            "종목코드",
            "회사명",
            "기업규모"
        ]
    ]
    .head(50)
    .to_string(index=False)
)


# ============================================================
# 15. 결과 저장
# ============================================================

df.to_pickle(
    OUT_PKL
)

company_year.to_csv(
    OUT_COMPANY,
    index=False,
    encoding="utf-8-sig"
)


print()
print("=" * 70)
print("저장 완료")
print("=" * 70)

print(
    f"전체 재무데이터:\n"
    f"{OUT_PKL}"
)

print(
    f"\n연도별 기업규모표:\n"
    f"{OUT_COMPANY}"
)

연도별 기업규모 리스트 불러오기

[2022]
대기업 : 2,875개
중견기업 : 3,220개
중복 : 203개
대기업 파일 : 대기업리스트(2022).xlsx
중견기업 파일 : 중견기업리스트(2022).xlsx
※ 대기업/중견기업 리스트에 동시에 있으면 '대기업'을 우선 적용

[2023]
대기업 : 3,060개
중견기업 : 3,310개
중복 : 213개
대기업 파일 : 대기업리스트(2023).xlsx
중견기업 파일 : 중견기업리스트(2023).xlsx
※ 대기업/중견기업 리스트에 동시에 있으면 '대기업'을 우선 적용

[2024]
대기업 : 3,300개
중견기업 : 3,554개
중복 : 249개
대기업 파일 : 대기업리스트(2024).xlsx
중견기업 파일 : 중견기업리스트(2024).xlsx
※ 대기업/중견기업 리스트에 동시에 있으면 '대기업'을 우선 적용

[2025]
대기업 : 3,285개
중견기업 : 3,892개
중복 : 280개
대기업 파일 : 대기업리스트(2025).xlsx
중견기업 파일 : 중견기업리스트(2025).xlsx
※ 대기업/중견기업 리스트에 동시에 있으면 '대기업'을 우선 적용

[2026]
대기업 : 3,519개
중견기업 : 3,912개
중복 : 332개
대기업 파일 : 대기업리스트(2026).xlsx
중견기업 파일 : 중견기업리스트(2026).xlsx
※ 대기업/중견기업 리스트에 동시에 있으면 '대기업'을 우선 적용

포괄손익계산서 불러오는 중
전체 행 수 : 1,829,248
회사 수 : 3,518
대상연도 : [2022, 2023, 2024, 2025, 2026]

연도×기업 판별 대상 : 14,071개

연도별 기업규모 분류 결과
  연도 기업규모  기업수
2022  대기업  212
2022 중견기업  737
2022 중소기업 1506
2023  대기업  298
2023 중견기업  781
2023 중소기업 1731
2024  대기업  315
2024 중견기업  805
2024 중소기업 1819
2025  대기업  311
20

In [34]:
%pip install openpyxl


     -------------------------------------- 250.9/250.9 KB 3.1 MB/s eta 0:00:00


You should consider upgrading via the 'c:\Users\USER\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [38]:
df.head()

,연도,분기,파일보고서종류,재무구분,원본파일명,재무제표종류,종목코드,회사명,시장구분,업종,...,전기 3분기 3개월,전기 3분기 누적,당기 반기 3개월,당기 반기 누적,전기 반기 3개월,전기 반기 누적,당기,회사명_정규화,기업규모,분류근거
0,2022,1,1분기보고서,별도,2022_1분기보고서_03_포괄손익계산서_20230718.txt,"포괄손익계산서, 기능별 분류(세후기타포괄손익) - 별도재무제표",[000020],동화약품,유가증권시장상장법인,212,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,동화약품,중견기업,2022 중견기업리스트
1,2022,1,1분기보고서,별도,2022_1분기보고서_03_포괄손익계산서_20230718.txt,"포괄손익계산서, 기능별 분류(세후기타포괄손익) - 별도재무제표",[000020],동화약품,유가증권시장상장법인,212,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,동화약품,중견기업,2022 중견기업리스트
2,2022,1,1분기보고서,별도,2022_1분기보고서_03_포괄손익계산서_20230718.txt,"포괄손익계산서, 기능별 분류(세후기타포괄손익) - 별도재무제표",[000020],동화약품,유가증권시장상장법인,212,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,동화약품,중견기업,2022 중견기업리스트
3,2022,1,1분기보고서,별도,2022_1분기보고서_03_포괄손익계산서_20230718.txt,"포괄손익계산서, 기능별 분류(세후기타포괄손익) - 별도재무제표",[000020],동화약품,유가증권시장상장법인,212,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,동화약품,중견기업,2022 중견기업리스트
4,2022,1,1분기보고서,별도,2022_1분기보고서_03_포괄손익계산서_20230718.txt,"포괄손익계산서, 기능별 분류(세후기타포괄손익) - 별도재무제표",[000020],동화약품,유가증권시장상장법인,212,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,동화약품,중견기업,2022 중견기업리스트


In [ ]:
df = pd.read_pickle(
    "datas/포괄손익계산서_2022_2026반기_기업규모분류.pkl"
)

df["기업규모"].value_counts()


KeyboardInterrupt: 

In [43]:
from pathlib import Path
import pandas as pd
import numpy as np
import re


# ============================================================
# 0. 파일 불러오기
# ============================================================

DATA_DIR = Path("datas")

# 기업규모(중소/중견/대기업)가 이미 붙어 있는 파일
# FILE_PATH = DATA_DIR / "포괄손익계산서_2022_2026반기_기업규모분류.pkl"

# df = pd.read_pickle(FILE_PATH)

print("원본 행 수:", f"{len(df):,}")
print("원본 열 수:", len(df.columns))


# ============================================================
# 1. 기본 정리
# ============================================================

d = df.copy()

# 컬럼명 공백 제거
d.columns = [
    str(c).strip()
    for c in d.columns
]


# 필수 열 확인
required = [
    "종목코드",
    "회사명",
    "결산기준일",
    "항목코드",
    "항목명",
    "기업규모"
]

missing = [
    c for c in required
    if c not in d.columns
]

if missing:
    raise ValueError(
        f"필수 컬럼이 없습니다: {missing}"
    )


# ============================================================
# 2. 보고서 종류 컬럼 확인
# ============================================================

if "파일보고서종류" in d.columns:
    REPORT_COL = "파일보고서종류"

elif "보고서종류" in d.columns:
    REPORT_COL = "보고서종류"

else:
    raise ValueError(
        "'파일보고서종류' 또는 '보고서종류' 컬럼이 없습니다."
    )


# ============================================================
# 3. 종목코드 / 날짜 정리
# ============================================================

d["종목코드"] = (
    d["종목코드"]
    .astype(str)
    .str.replace("[", "", regex=False)
    .str.replace("]", "", regex=False)
    .str.strip()
)


d["결산기준일"] = pd.to_datetime(
    d["결산기준일"],
    errors="coerce"
)


# 실제 달력 기준 연도/분기
#
# 이게 중요함.
# 예를 들어 결산기준일이 2025-06-30이면
# 실제 분석 시점은 2025Q2로 처리

d["기준연도"] = (
    d["결산기준일"]
    .dt.year
)

d["기준분기"] = (
    d["결산기준일"]
    .dt.quarter
)


# ============================================================
# 4. 연결재무제표 우선순위
# ============================================================

if "재무구분" in d.columns:

    d["재무우선순위"] = (
        d["재무구분"]
        .map({
            "연결": 0,
            "별도": 1
        })
        .fillna(2)
    )

else:

    # 재무구분 열이 없다면 재무제표종류에서 판단
    d["재무우선순위"] = np.where(
        d["재무제표종류"]
        .astype(str)
        .str.contains("연결"),
        0,
        1
    )


# ============================================================
# 5. 매출액 / 영업이익 항목만 골라내기
# ============================================================

def classify_account(row):

    code = str(
        row["항목코드"]
    ).strip()

    name = str(
        row["항목명"]
    ).strip()


    # -------------------------------
    # 영업이익
    # -------------------------------

    if code == "dart_OperatingIncomeLoss":
        return "영업이익"

    if name in [
        "영업이익",
        "영업이익(손실)",
        "영업손익",
        "영업이익(손실액)"
    ]:
        return "영업이익"


    # -------------------------------
    # 매출
    # -------------------------------

    if code == "ifrs-full_Revenue":
        return "매출액"

    if name in [
        "매출액",
        "영업수익",
        "수익(매출액)",
        "매출",
        "영업수익(매출액)"
    ]:
        return "매출액"


    return None


d["계정"] = d.apply(
    classify_account,
    axis=1
)


d = d[
    d["계정"].isin(
        [
            "매출액",
            "영업이익"
        ]
    )
].copy()


print()
print(
    "매출/영업이익 행 수:",
    f"{len(d):,}"
)


# ============================================================
# 6. 숫자 변환 함수
# ============================================================

def to_num(x):

    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    if x == "":
        return np.nan

    x = (
        x
        .replace(",", "")
        .replace(" ", "")
    )

    # (123) -> -123
    if (
        x.startswith("(")
        and x.endswith(")")
    ):

        x = "-" + x[1:-1]


    return pd.to_numeric(
        x,
        errors="coerce"
    )


# ============================================================
# 7. 실제 컬럼명 찾기
# ============================================================

def find_col(
    columns,
    must_include
):

    for col in columns:

        text = re.sub(
            r"\s+",
            "",
            str(col)
        )

        if all(
            re.sub(r"\s+", "", x) in text
            for x in must_include
        ):

            return col

    return None


cols = d.columns.tolist()


# -------------------------------
# 1분기
# -------------------------------

Q1_3M = find_col(
    cols,
    [
        "당기",
        "1분기",
        "3개월"
    ]
)

Q1_CUM = find_col(
    cols,
    [
        "당기",
        "1분기",
        "누적"
    ]
)


# -------------------------------
# 반기
# -------------------------------

Q2_3M = find_col(
    cols,
    [
        "당기",
        "반기",
        "3개월"
    ]
)

Q2_CUM = find_col(
    cols,
    [
        "당기",
        "반기",
        "누적"
    ]
)


# -------------------------------
# 3분기
# -------------------------------

Q3_3M = find_col(
    cols,
    [
        "당기",
        "3분기",
        "3개월"
    ]
)

Q3_CUM = find_col(
    cols,
    [
        "당기",
        "3분기",
        "누적"
    ]
)


# -------------------------------
# 사업보고서 당기
# -------------------------------

ANNUAL = None

for col in cols:

    if str(col).strip() == "당기":
        ANNUAL = col
        break


print()
print("=" * 60)
print("사용할 재무 컬럼 확인")
print("=" * 60)

print("1분기 3개월 :", Q1_3M)
print("1분기 누적   :", Q1_CUM)

print("반기 3개월   :", Q2_3M)
print("반기 누적    :", Q2_CUM)

print("3분기 3개월  :", Q3_3M)
print("3분기 누적   :", Q3_CUM)

print("사업보고서 당기 :", ANNUAL)


# ============================================================
# 8. 연결 우선으로 한 기업/계정 한 값만 선택하는 함수
# ============================================================

def select_one_statement(temp):

    temp = temp.copy()

    temp = temp.sort_values(
        "재무우선순위"
    )

    temp = temp.drop_duplicates(
        subset=[
            "결산기준일",
            "종목코드",
            "계정"
        ],
        keep="first"
    )

    return temp


# ============================================================
# 9. Q1 만들기
# ============================================================

q1 = d[
    d[REPORT_COL]
    .astype(str)
    .str.contains("1분기")
].copy()


if Q1_3M is not None:

    q1["값"] = (
        q1[Q1_3M]
        .apply(to_num)
    )

elif Q1_CUM is not None:

    # 1분기는 3개월 = 누적이라 사용 가능
    q1["값"] = (
        q1[Q1_CUM]
        .apply(to_num)
    )

else:

    raise ValueError(
        "1분기 값을 찾지 못했습니다."
    )


q1 = select_one_statement(q1)


# ============================================================
# 10. Q2 만들기
# ============================================================

q2 = d[
    d[REPORT_COL]
    .astype(str)
    .str.contains("반기")
].copy()


if Q2_3M is None:

    raise ValueError(
        "'당기 반기 3개월' 컬럼을 찾지 못했습니다."
    )


q2["값"] = (
    q2[Q2_3M]
    .apply(to_num)
)


q2 = select_one_statement(q2)


# ============================================================
# 11. Q3 만들기
# ============================================================

q3 = d[
    d[REPORT_COL]
    .astype(str)
    .str.contains("3분기")
].copy()


if Q3_3M is None:

    raise ValueError(
        "'당기 3분기 3개월' 컬럼을 찾지 못했습니다."
    )


q3["값"] = (
    q3[Q3_3M]
    .apply(to_num)
)


q3 = select_one_statement(q3)


# ============================================================
# 12. Q4 계산
#
# Q4 =
# 사업보고서 연간값 - 3분기 누적값
#
# 중요:
# 12월 결산기업만 있다고 가정하지 않고
# 결산기준일을 이용해서 실제 3개월 전 Q3를 찾음
# ============================================================


# -------------------------------
# 사업보고서
# -------------------------------

annual = d[
    d[REPORT_COL]
    .astype(str)
    .str.contains("사업보고서")
].copy()


if ANNUAL is None:

    raise ValueError(
        "사업보고서의 '당기' 컬럼을 찾지 못했습니다."
    )


annual["연간값"] = (
    annual[ANNUAL]
    .apply(to_num)
)


annual = select_one_statement(
    annual
)


# -------------------------------
# 3분기 누적
# -------------------------------

q3_cum = d[
    d[REPORT_COL]
    .astype(str)
    .str.contains("3분기")
].copy()


if Q3_CUM is None:

    raise ValueError(
        "'당기 3분기 누적' 컬럼을 찾지 못했습니다."
    )


q3_cum["3분기누적값"] = (
    q3_cum[Q3_CUM]
    .apply(to_num)
)


q3_cum = select_one_statement(
    q3_cum
)


# ============================================================
# 13. 사업보고서와 그 직전 3분기 연결
#
# 연말 결산뿐 아니라
# 3월/6월/9월 결산기업도 대응
# ============================================================

annual["결산월_기간"] = (
    annual["결산기준일"]
    .dt.to_period("M")
)


# 사업연도 종료월에서 3개월 전
annual["예상_Q3월"] = (
    annual["결산월_기간"] - 3
)


q3_cum["Q3월"] = (
    q3_cum["결산기준일"]
    .dt.to_period("M")
)


q4 = annual.merge(
    q3_cum[
        [
            "종목코드",
            "계정",
            "Q3월",
            "3분기누적값"
        ]
    ],

    left_on=[
        "종목코드",
        "계정",
        "예상_Q3월"
    ],

    right_on=[
        "종목코드",
        "계정",
        "Q3월"
    ],

    how="inner"
)


q4["값"] = (
    q4["연간값"]
    - q4["3분기누적값"]
)


# ============================================================
# 14. 필요한 열만 통일
# ============================================================

KEEP = [
    "결산기준일",
    "기준연도",
    "기준분기",
    "종목코드",
    "회사명",
    "기업규모",
    "계정",
    "값"
]


q1 = q1[KEEP].copy()
q2 = q2[KEEP].copy()
q3 = q3[KEEP].copy()
q4 = q4[KEEP].copy()


# ============================================================
# 15. 전체 분기 데이터 합치기
# ============================================================

quarter_long = pd.concat(
    [
        q1,
        q2,
        q3,
        q4
    ],
    ignore_index=True
)


print()
print(
    "분기 계정 데이터:",
    f"{len(quarter_long):,}"
)


# ============================================================
# 16. 매출액 / 영업이익을 한 행에 배치
# ============================================================

quarter = (
    quarter_long
    .pivot_table(

        index=[
            "결산기준일",
            "기준연도",
            "기준분기",
            "종목코드",
            "회사명",
            "기업규모"
        ],

        columns="계정",

        values="값",

        aggfunc="first"
    )
    .reset_index()
)


quarter.columns.name = None


# ============================================================
# 17. 매출과 영업이익이 둘 다 존재하는 기업만
# ============================================================

quarter = quarter[
    quarter["매출액"].notna()
    &
    quarter["영업이익"].notna()
].copy()


# 매출 <= 0 제외
quarter = quarter[
    quarter["매출액"] > 0
].copy()


# ============================================================
# 18. 영업이익률 계산
# ============================================================

quarter["영업이익률"] = (
    quarter["영업이익"]
    / quarter["매출액"]
    * 100
)


# 무한대 제거
quarter["영업이익률"] = (
    quarter["영업이익률"]
    .replace(
        [
            np.inf,
            -np.inf
        ],
        np.nan
    )
)


quarter = quarter.dropna(
    subset=["영업이익률"]
).copy()


print()
print(
    "영업이익률 계산 기업-분기:",
    f"{len(quarter):,}"
)


# ============================================================
# 19. ★★★ IQR 이상치 제거 ★★★
#
# 여기서 먼저 제거!
#
# 각 연도 × 분기 전체기업을 대상으로:
#
# Q1 = 25%
# Q3 = 75%
# IQR = Q3-Q1
#
# 하한 = Q1 - 1.5*IQR
# 상한 = Q3 + 1.5*IQR
#
# 중소/비중소로 나누기 전에 제거한다.
# ============================================================

clean_parts = []
iqr_records = []


for (
    year,
    q
), g in quarter.groupby(
    [
        "기준연도",
        "기준분기"
    ]
):

    g = g.copy()

    q1_value = (
        g["영업이익률"]
        .quantile(0.25)
    )

    q3_value = (
        g["영업이익률"]
        .quantile(0.75)
    )

    iqr = (
        q3_value
        - q1_value
    )

    lower = (
        q1_value
        - 1.5 * iqr
    )

    upper = (
        q3_value
        + 1.5 * iqr
    )


    before = len(g)


    # -------------------------------
    # IQR 범위 안에 있는 기업만
    # -------------------------------

    cleaned = g[
        g["영업이익률"]
        .between(
            lower,
            upper,
            inclusive="both"
        )
    ].copy()


    after = len(cleaned)


    clean_parts.append(
        cleaned
    )


    iqr_records.append({

        "연도": int(year),
        "분기": int(q),

        "Q1": q1_value,
        "Q3": q3_value,
        "IQR": iqr,

        "하한": lower,
        "상한": upper,

        "제거전": before,
        "제거후": after,
        "제거수": before - after,

        "제거율": (
            (before - after)
            / before
            * 100
            if before > 0
            else np.nan
        )
    })


# ============================================================
# 20. 이상치 제거 데이터
# ============================================================

quarter_clean = pd.concat(
    clean_parts,
    ignore_index=True
)


iqr_summary = pd.DataFrame(
    iqr_records
)


print()
print("=" * 70)
print("IQR 이상치 제거 결과")
print("=" * 70)

print(
    iqr_summary
    .round(2)
    .to_string(
        index=False
    )
)


print()
print(
    "IQR 제거 전:",
    f"{len(quarter):,}"
)

print(
    "IQR 제거 후:",
    f"{len(quarter_clean):,}"
)


# ============================================================
# 21. 이제 중소기업 / 비중소기업 구분
#
# 중견 + 대기업 = 비중소기업
# ============================================================

quarter_clean[
    "기업규모_2분류"
] = quarter_clean[
    "기업규모"
].map({

    "중소기업": "중소기업",

    "중견기업": "비중소기업",

    "대기업": "비중소기업"
})


# 분류 안 된 기업 제외
quarter_clean = quarter_clean[
    quarter_clean[
        "기업규모_2분류"
    ].notna()
].copy()


# ============================================================
# 22. 사건 전년 동분기 비교 함수
# ============================================================

def compare_event(
    data,
    event,
    current_year,
    q
):

    previous_year = (
        current_year - 1
    )


    # ========================================================
    # 현재 시점
    # ========================================================

    current = data[
        (
            data["기준연도"]
            == current_year
        )
        &
        (
            data["기준분기"]
            == q
        )
    ][
        [
            "종목코드",
            "회사명",
            "기업규모",
            "기업규모_2분류",
            "영업이익률"
        ]
    ].copy()


    current = current.rename(
        columns={
            "영업이익률":
                "당기_영업이익률"
        }
    )


    # 같은 기업 중복 방지
    current = current.drop_duplicates(
        subset=[
            "종목코드"
        ],
        keep="first"
    )


    # ========================================================
    # 전년 동분기
    # ========================================================

    previous = data[
        (
            data["기준연도"]
            == previous_year
        )
        &
        (
            data["기준분기"]
            == q
        )
    ][
        [
            "종목코드",
            "회사명",
            "영업이익률"
        ]
    ].copy()


    previous = previous.rename(
        columns={
            "영업이익률":
                "전년_영업이익률"
        }
    )


    previous = previous.drop_duplicates(
        subset=[
            "종목코드"
        ],
        keep="first"
    )


    # ========================================================
    # 같은 기업만 비교
    # ========================================================

    paired = current.merge(
        previous[
            [
                "종목코드",
                "전년_영업이익률"
            ]
        ],

        on="종목코드",

        how="inner"
    )


    # ========================================================
    # 영업이익률 변화
    # ========================================================

    paired[
        "영업이익률_변화"
    ] = (

        paired[
            "당기_영업이익률"
        ]

        -

        paired[
            "전년_영업이익률"
        ]
    )


    # ========================================================
    # 중소 vs 비중소 집계
    # ========================================================

    result = (
        paired
        .groupby(
            "기업규모_2분류"
        )
        .agg(

            기업수=(
                "종목코드",
                "nunique"
            ),

            전년_영업이익률=(
                "전년_영업이익률",
                "mean"
            ),

            당기_영업이익률=(
                "당기_영업이익률",
                "mean"
            ),

            평균_변화=(
                "영업이익률_변화",
                "mean"
            ),

            중앙값_변화=(
                "영업이익률_변화",
                "median"
            ),

            악화_기업_비율=(
                "영업이익률_변화",

                lambda x:
                    (
                        x < 0
                    ).mean()
                    * 100
            ),

            변화_표준편차=(
                "영업이익률_변화",
                "std"
            )
        )
        .reset_index()
    )


    # 사건 정보
    result.insert(
        0,
        "사건",
        event
    )


    result.insert(
        1,
        "비교기간",
        (
            f"{current_year}Q{q}"
            f" vs "
            f"{previous_year}Q{q}"
        )
    )


    return (
        result,
        paired
    )


# ============================================================
# 23. 분석할 네 시기
# ============================================================

events = [

    # 홍해
    (
        "홍해",
        2023,
        4
    ),

    (
        "홍해",
        2024,
        1
    ),

    # 호르무즈
    (
        "호르무즈",
        2026,
        1
    ),

    (
        "호르무즈",
        2026,
        2
    )
]


# ============================================================
# 24. 전체 분석
# ============================================================

results = []

paired_dict = {}


for (
    event,
    year,
    q
) in events:


    result, paired = (
        compare_event(

            quarter_clean,

            event,

            year,

            q
        )
    )


    results.append(
        result
    )


    paired_dict[
        f"{event}_{year}Q{q}"
    ] = paired


# ============================================================
# 25. 결과 합치기
# ============================================================

final_result = pd.concat(
    results,
    ignore_index=True
)


# ============================================================
# 26. 반올림
# ============================================================

numeric_cols = [

    "전년_영업이익률",

    "당기_영업이익률",

    "평균_변화",

    "중앙값_변화",

    "악화_기업_비율",

    "변화_표준편차"
]


final_result[
    numeric_cols
] = final_result[
    numeric_cols
].round(2)


# ============================================================
# 27. 최종 결과
# ============================================================

print()
print("=" * 90)
print("중소기업 vs 비중소기업 영업이익률 변화")
print("=" * 90)

print(
    final_result
    .to_string(
        index=False
    )
)


# ============================================================
# 28. 결과 저장
# ============================================================

quarter_clean.to_pickle(
    DATA_DIR
    / "분기영업이익률_IQR이상치제거.pkl"
)


iqr_summary.to_csv(
    DATA_DIR
    / "IQR_이상치제거현황.csv",

    index=False,

    encoding="utf-8-sig"
)


final_result.to_csv(
    DATA_DIR
    / "사건별_중소_vs_비중소_영업이익률.csv",

    index=False,

    encoding="utf-8-sig"
)


print()
print("저장 완료!")

원본 행 수: 1,829,248
원본 열 수: 35

매출/영업이익 행 수: 155,057

사용할 재무 컬럼 확인
1분기 3개월 : 당기 1분기 3개월
1분기 누적   : 당기 1분기 누적
반기 3개월   : 당기 반기 3개월
반기 누적    : 당기 반기 누적
3분기 3개월  : 당기 3분기 3개월
3분기 누적   : 당기 3분기 누적
사업보고서 당기 : 당기

분기 계정 데이터: 82,060

영업이익률 계산 기업-분기: 39,976

IQR 이상치 제거 결과
  연도  분기      Q1    Q3    IQR      하한     상한  제거전  제거후  제거수   제거율
2022   1   -3.34  9.95  13.29  -23.27  29.89 2020 1722  298 14.75
2022   2   -2.67 10.63  13.30  -22.61  30.58 2066 1754  312 15.10
2022   3   -3.86  9.50  13.35  -23.89  29.53 2096 1791  305 14.55
2022   4   -5.91  8.39  14.30  -27.36  29.84 2062 1743  319 15.47
2023   1   -7.06  7.52  14.58  -28.93  29.38 2132 1827  305 14.31
2023   2   -6.29  8.72  15.00  -28.79  31.23 2156 1853  303 14.05
2023   3   -6.71  8.45  15.16  -29.46  31.19 2207 1895  312 14.14
2023   4   -8.22  7.97  16.19  -32.50  32.25 2163 1839  324 14.98
2024   1   -7.83  7.42  15.25  -30.70  30.29 2249 1919  330 14.67
2024   2   -6.03  8.69  14.72  -28.10  30.77 2248 1903  345 15.35
2024   3   